## 1. Imports and Setup

In [1]:
import os
import re
import json
from pathlib import Path
from dataclasses import dataclass, replace
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple, Iterable

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping, Polygon
from shapely.affinity import translate
import networkx as nx
from sklearn.neighbors import NearestNeighbors

SKIMAGE_AVAILABLE = True
try:
    from skimage.registration import phase_cross_correlation
    from skimage.transform import resize
except Exception as e:
    SKIMAGE_AVAILABLE = False
    print("⚠️ scikit-image not available. Phase correlation alignment will be skipped.")
    print(str(e))

In [2]:
@dataclass
class MatchCaseConfig:
    name: str
    base_similarity_weights: Dict[str, float]
    scoring_weights: Dict[str, float]
    similarity_threshold: float
    min_iou: float = 0.0
    min_overlap_prev: float = 0.0
    min_overlap_curr: float = 0.0
    max_centroid_dist: Optional[float] = None
    mutual_best: bool = False
    allow_multiple: bool = False
    max_edges_per_prev: Optional[int] = None
    max_edges_per_curr: Optional[int] = None


class TreeTrackingGraph:
    """
    Tree crown tracker across orthomosaics using a directed graph with ultra-relaxed parameters.
    
    Supports automatic file discovery, image extraction, spatial alignment, and configurable matching.
    """
    
    def __init__(
        self,
        crown_dir: Optional[str] = None,
        ortho_dir: Optional[str] = None,
        output_dir: str = '../../output',
        simplify_tol: float = 1.0,
        max_crowns_preview: int = 200,
        auto_discover: bool = True
    ) -> None:
        self.output_dir = output_dir
        self.simplify_tol = simplify_tol
        self.max_crowns_preview = max_crowns_preview
        self.crown_dir = crown_dir or self._resolve_dir('input/input_crowns', '../../input/input_crowns')
        self.ortho_dir = ortho_dir or self._resolve_dir('input/input_om', '../../input/input_om')
        
        self.file_pairs: List[Tuple[str, Optional[str]]] = []
        self.om_ids: List[int] = []
        self.crowns_gdfs: Dict[int, gpd.GeoDataFrame] = {}
        self.crown_attrs: Dict[int, List[Dict[str, Any]]] = {}
        self.crown_images: Dict[int, List[Optional[np.ndarray]]] = {}
        self.crown_crs: Dict[int, Optional[Any]] = {}
        self.ortho_crs: Dict[int, Optional[Any]] = {}
        self.G: nx.DiGraph = nx.DiGraph()
        self.match_case_mode: str = "balanced"
        
        # Use ultra-relaxed configs by default
        self.case_configs: Dict[str, MatchCaseConfig] = self._ultra_relaxed_case_configs()
        self.case_order: List[str] = ['one_to_one', 'containment', 'nearby']
        self.last_case_counts: Dict[str, int] = {}
        self.last_selected_counts: Dict[str, int] = {}
        
        if auto_discover:
            self.discover_files()
    
    @staticmethod
    def _resolve_dir(root_rel: str, nb_rel: str) -> str:
        """Resolve directory path from multiple possible locations."""
        candidates = [
            os.path.abspath(os.path.join(os.getcwd(), root_rel)),
            os.path.abspath(os.path.join(os.getcwd(), nb_rel)),
        ]
        for p in candidates:
            if os.path.isdir(p):
                return p
        raise FileNotFoundError(f"Could not resolve directory for {root_rel}. Tried: {candidates}")
    
    @staticmethod
    def _extract_numeric_id(name: str) -> Optional[int]:
        """Extract numeric ID from filename."""
        m = re.search(r"(\d+)", os.path.basename(name))
        return int(m.group(1)) if m else None
    
    def discover_files(self) -> None:
        """Automatically discover and pair crown GeoPackages with orthomosaics."""
        crown_files = [
            os.path.join(self.crown_dir, f) 
            for f in os.listdir(self.crown_dir) 
            if f.lower().endswith('.gpkg')
        ]
        ortho_files = [
            os.path.join(self.ortho_dir, f) 
            for f in os.listdir(self.ortho_dir) 
            if f.lower().endswith('.tif')
        ] if os.path.exists(self.ortho_dir) else []
        
        if not crown_files:
            raise FileNotFoundError(f"No .gpkg crown files found in {self.crown_dir}")
        
        # Group files by numeric ID
        crowns_by_id = {}
        for cf in crown_files:
            cid = self._extract_numeric_id(cf)
            crowns_by_id[cid if cid is not None else cf] = cf
        
        orthos_by_id = {}
        for of in ortho_files:
            oid = self._extract_numeric_id(of)
            orthos_by_id[oid if oid is not None else of] = of
        
        # Find matching numeric IDs
        numeric_ids = sorted(
            set(k for k in crowns_by_id.keys() if isinstance(k, int)) & 
            set(k for k in orthos_by_id.keys() if isinstance(k, int))
        )
        
        # Pair files
        file_pairs: List[Tuple[str, Optional[str]]] = []
        if numeric_ids:
            for nid in numeric_ids:
                file_pairs.append((crowns_by_id[nid], orthos_by_id.get(nid)))
            crown_only = sorted(
                k for k in crowns_by_id.keys() 
                if isinstance(k, int) and k not in numeric_ids
            )
            for nid in crown_only:
                file_pairs.append((crowns_by_id[nid], None))
        else:
            crown_files_sorted = sorted(crown_files)
            ortho_files_sorted = sorted(ortho_files)
            for i, cf in enumerate(crown_files_sorted):
                of = ortho_files_sorted[i] if i < len(ortho_files_sorted) else None
                file_pairs.append((cf, of))
        
        # Extract OM IDs
        om_ids: List[int] = []
        for cf, _ in file_pairs:
            cid = self._extract_numeric_id(cf)
            om_ids.append(cid if cid is not None else len(om_ids) + 1)
        
        # Sort by OM ID
        pairs_with_id = sorted(
            [(oid, cf, of) for oid, (cf, of) in zip(om_ids, file_pairs)],
            key=lambda x: x[0]
        )
        self.file_pairs = [(cf, of) for _, cf, of in pairs_with_id]
        self.om_ids = [oid for oid, _, _ in pairs_with_id]
    
    def load_data(
        self,
        load_images: bool = False,
        align: bool = False,
        reference_om_id: Optional[int] = None
    ) -> None:
        """
        Load crown data from GeoPackages and optionally extract image patches.
        
        CRITICAL: Images are extracted from ORIGINAL geometries BEFORE alignment.
        This ensures image-geometry correspondence is preserved.
        
        Args:
            load_images: If True, extract RGB image patches from orthomosaics for each crown
            align: If True, align all OMs to reference OM using translational shift
            reference_om_id: OM ID to use as alignment reference (default: first OM)
        """
        self.crowns_gdfs.clear()
        self.crown_attrs.clear()
        self.crown_images.clear()
        self.crown_crs.clear()
        self.ortho_crs.clear()
        
        print(f"\n{'='*70}")
        print("LOADING DATA")
        print(f"{'='*70}")
        
        # Step 1: Load geometries and extract images from ORIGINAL positions
        for om_id, (crown_file, ortho_file) in zip(self.om_ids, self.file_pairs):
            print(f"\nOM{om_id}: Loading {os.path.basename(crown_file)}")
            gdf = gpd.read_file(crown_file)
            self.crowns_gdfs[om_id] = gdf
            self.crown_crs[om_id] = gdf.crs
            
            # Extract images from ORIGINAL geometries BEFORE any shifting
            if load_images and ortho_file and os.path.exists(ortho_file):
                print(f"  Extracting {len(gdf)} crown images from {os.path.basename(ortho_file)}...")
                with rasterio.open(ortho_file) as src:
                    self.ortho_crs[om_id] = src.crs
                    patches: List[Optional[np.ndarray]] = []
                    for _, row in gdf.iterrows():
                        geom = [mapping(row.geometry)]
                        try:
                            out_image, _ = mask(src, geom, crop=True)
                            img_patch = np.moveaxis(out_image, 0, -1)  # (C, H, W) -> (H, W, C)
                        except Exception:
                            img_patch = None
                        patches.append(img_patch)
                self.crown_images[om_id] = patches
                print(f"  ✓ Extracted {sum(1 for p in patches if p is not None)} images")
            else:
                self.crown_images[om_id] = [None] * len(gdf)
                if ortho_file and os.path.exists(ortho_file):
                    with rasterio.open(ortho_file) as src:
                        self.ortho_crs[om_id] = src.crs
                else:
                    self.ortho_crs[om_id] = None
            
            # Compute attributes from original geometries
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in gdf.iterrows()
            ]
        
        self._check_crs_consistency()
        print(f"\n{'='*70}")
        
        # Step 2: Apply alignment if requested (geometries shift, images stay with original crowns)
        if align:
            print(f"\n{'='*70}")
            print("APPLYING SPATIAL ALIGNMENT")
            print(f"{'='*70}")
            self.alignment_shifts = self.align_to_reference(reference_om_id=reference_om_id)
            print(f"{'='*70}\n")
    
    def _check_crs_consistency(self) -> None:
        """Check CRS consistency across crown files and orthomosaics."""
        print("CRS CONSISTENCY CHECK")
        print("-" * 60)
        crown_crs_values = {om_id: crs for om_id, crs in self.crown_crs.items()}
        ortho_crs_values = {om_id: crs for om_id, crs in self.ortho_crs.items()}
        crown_crs_set = {str(crs) for crs in crown_crs_values.values() if crs is not None}
        if not crown_crs_set:
            print("⚠️  Crown CRS: None detected in all crown files")
        elif len(crown_crs_set) > 1:
            print("⚠️  Crown CRS mismatch across OMs:")
            for om_id, crs in crown_crs_values.items():
                print(f"  OM{om_id}: {crs}")
        else:
            print(f"✓ Crown CRS consistent: {next(iter(crown_crs_set))}")
        
        for om_id in self.om_ids:
            crown_crs = crown_crs_values.get(om_id)
            ortho_crs = ortho_crs_values.get(om_id)
            if crown_crs is None:
                print(f"⚠️  OM{om_id}: crown CRS is None")
            if ortho_crs is None:
                print(f"⚠️  OM{om_id}: ortho CRS is None or ortho missing")
            if crown_crs is not None and ortho_crs is not None and crown_crs != ortho_crs:
                print(f"⚠️  OM{om_id}: crown CRS != ortho CRS ({crown_crs} vs {ortho_crs})")
        print("-" * 60)
    
    def align_to_reference(
        self,
        reference_om_id: Optional[int] = None,
        threshold: float = 20.0,
        min_matches: int = 5
    ) -> Dict[int, Tuple[float, float]]:
        """
        Align all OMs to a reference OM using mutual nearest neighbors and robust median shift.
        
        Args:
            reference_om_id: OM ID to use as reference (default: first OM)
            threshold: Maximum distance for considering a match (meters)
            min_matches: Minimum number of matches required
        
        Returns:
            Dictionary mapping om_id -> (dx, dy) shift applied
        """
        if not self.crowns_gdfs:
            raise ValueError("No crown data loaded. Call load_data() first.")
        
        if reference_om_id is None:
            reference_om_id = self.om_ids[0]
        if reference_om_id not in self.crowns_gdfs:
            raise ValueError(f"Reference OM {reference_om_id} not found in loaded data.")
        
        ref = self.crowns_gdfs[reference_om_id]
        ref_centroids = np.array([[g.centroid.x, g.centroid.y] for g in ref.geometry])
        shifts = {reference_om_id: (0.0, 0.0)}
        
        print(f"Aligning all OMs to reference OM{reference_om_id}")
        print(f"  Reference crowns: {len(ref_centroids)}")
        print()
        
        nn_ref = NearestNeighbors(n_neighbors=1).fit(ref_centroids)

        for om_id in self.om_ids:
            if om_id == reference_om_id:
                continue
            curr = self.crowns_gdfs[om_id].copy()
            curr_centroids = np.array([[g.centroid.x, g.centroid.y] for g in curr.geometry])
            if len(curr_centroids) == 0:
                print(f"OM{om_id}: no crowns, skipping alignment")
                shifts[om_id] = (0.0, 0.0)
                continue
            
            dist_to_ref, idx_ref = nn_ref.kneighbors(curr_centroids)
            nn_curr = NearestNeighbors(n_neighbors=1).fit(curr_centroids)
            dist_to_curr, idx_curr = nn_curr.kneighbors(ref_centroids)
            
            mutual = []
            for i, (d, ref_idx) in enumerate(zip(dist_to_ref[:, 0], idx_ref[:, 0])):
                if d < threshold and idx_curr[ref_idx][0] == i and dist_to_curr[ref_idx][0] < threshold:
                    mutual.append((ref_idx, i, d))
            
            if len(mutual) < min_matches:
                relaxed = [(idx_ref[i][0], i, dist_to_ref[i][0]) for i in range(len(dist_to_ref)) if dist_to_ref[i][0] < threshold * 1.5]
                mutual = mutual + relaxed
            
            if len(mutual) < min_matches:
                print(f"OM{om_id}: insufficient matches ({len(mutual)}/{min_matches}), skipping alignment")
                shifts[om_id] = (0.0, 0.0)
                continue
            
            shifts_array = np.array([ref_centroids[a] - curr_centroids[b] for a, b, _ in mutual])
            median_shift = np.median(shifts_array, axis=0)
            residuals = np.linalg.norm(shifts_array - median_shift, axis=1)
            median_resid = float(np.median(residuals)) if len(residuals) else 0.0
            mad = float(np.median(np.abs(residuals - median_resid))) or 1e-6
            inliers = residuals <= median_resid + 3.0 * mad
            
            if not np.any(inliers):
                inliers = np.ones_like(residuals, dtype=bool)
            
            dx, dy = np.median(shifts_array[inliers], axis=0)
            print(f"OM{om_id}: matches={len(mutual)}, inliers={np.sum(inliers)}, dx={dx:.2f}, dy={dy:.2f}, med_res={median_resid:.2f}")
            
            curr['geometry'] = curr['geometry'].apply(lambda g: translate(g, xoff=dx, yoff=dy))
            self.crowns_gdfs[om_id] = curr
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in curr.iterrows()
            ]
            shifts[om_id] = (float(dx), float(dy))
        
        return shifts
    
    @staticmethod
    def _compute_crown_attributes(geometry) -> Dict[str, Any]:
        """Compute geometric attributes for a crown polygon."""
        centroid = geometry.centroid
        area = geometry.area
        perimeter = geometry.length
        bounds = geometry.bounds
        compactness = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
        
        try:
            min_rect = geometry.minimum_rotated_rectangle
            coords = list(min_rect.exterior.coords)
            side1 = np.linalg.norm(np.array(coords[0]) - np.array(coords[1]))
            side2 = np.linalg.norm(np.array(coords[1]) - np.array(coords[2]))
            major_axis = max(side1, side2)
            minor_axis = min(side1, side2)
            eccentricity = minor_axis / major_axis if major_axis > 0 else 1
        except Exception:
            eccentricity = 1
        
        aspect_ratio = (bounds[3] - bounds[1]) / (bounds[2] - bounds[0]) if bounds[2] != bounds[0] else 1
        
        return {
            'geometry': geometry,
            'centroid': centroid,
            'area': area,
            'perimeter': perimeter,
            'compactness': compactness,
            'eccentricity': eccentricity,
            'aspect_ratio': aspect_ratio,
            'bounds': bounds,
        }
    
    def _ultra_relaxed_case_configs(self) -> Dict[str, MatchCaseConfig]:
        """
        ULTRA RELAXED matching parameters - prioritizes spatial proximity over shape similarity.
        
        Designed to bridge difficult OM transitions with minimal overlap or shape changes.
        """
        return {
            'one_to_one': MatchCaseConfig(
                name='one_to_one',
                base_similarity_weights={
                    'spatial': 0.50,  # Strong emphasis on proximity
                    'area': 0.15,
                    'shape': 0.05,    # De-emphasize shape
                    'iou': 0.30
                },
                scoring_weights={
                    'base': 0.60, 
                    'iou': 0.10, 
                    'overlap_prev': 0.05, 
                    'overlap_curr': 0.05, 
                    'centroid': 0.20
                },
                similarity_threshold=0.25,
                min_iou=0.01,
                min_overlap_prev=0.10,
                min_overlap_curr=0.10,
                max_centroid_dist=50.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=3,
                max_edges_per_curr=3,
            ),
            'containment': MatchCaseConfig(
                name='containment',
                base_similarity_weights={
                    'spatial': 0.50,
                    'area': 0.20,
                    'shape': 0.10,
                    'iou': 0.20
                },
                scoring_weights={
                    'base': 0.40, 
                    'overlap_prev': 0.30, 
                    'overlap_curr': 0.30
                },
                similarity_threshold=0.25,
                min_iou=0.0,
                min_overlap_prev=0.25,
                min_overlap_curr=0.25,
                max_centroid_dist=150.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
            'nearby': MatchCaseConfig(
                name='nearby',
                base_similarity_weights={
                    'spatial': 0.85,  # Almost purely spatial
                    'area': 0.10,
                    'shape': 0.03,
                    'iou': 0.02
                },
                scoring_weights={
                    'base': 0.70, 
                    'centroid': 0.30
                },
                similarity_threshold=0.20,
                min_iou=0.0,
                min_overlap_prev=0.05,
                min_overlap_curr=0.05,
                max_centroid_dist=200.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
        }
    
    @staticmethod
    def _compute_iou(g1, g2) -> float:
        """Compute intersection over union for two geometries."""
        try:
            intersection = g1.intersection(g2).area
            union = g1.union(g2).area
        except Exception:
            intersection = 0.0
            union = g1.area + g2.area
        return intersection / union if union > 0 else 0.0
    
    def _weighted_similarity(
        self,
        a1: Dict[str, Any],
        a2: Dict[str, Any],
        weights: Optional[Dict[str, float]] = None,
        max_dist: float = 100.0
    ) -> Tuple[float, Dict[str, float]]:
        """Compute weighted similarity score between two crown attributes."""
        if weights is None:
            weights = {'spatial': 0.4, 'area': 0.2, 'shape': 0.2, 'iou': 0.2}
        
        # Spatial similarity
        centroid_dist = a1['centroid'].distance(a2['centroid'])
        spatial_sim = max(0.0, 1.0 - (centroid_dist / max_dist))
        
        # Area similarity
        area_sim = min(a1['area'], a2['area']) / max(a1['area'], a2['area']) if max(a1['area'], a2['area']) > 0 else 0.0
        
        # Shape similarity
        compactness_sim = 1.0 - abs(a1['compactness'] - a2['compactness'])
        eccentricity_sim = 1.0 - abs(a1['eccentricity'] - a2['eccentricity'])
        shape_sim = (compactness_sim + eccentricity_sim) / 2.0
        
        # IoU similarity
        iou_sim = self._compute_iou(a1['geometry'], a2['geometry'])
        
        # Weighted total
        total = (
            weights.get('spatial', 0.0) * spatial_sim +
            weights.get('area', 0.0) * area_sim +
            weights.get('shape', 0.0) * shape_sim +
            weights.get('iou', 0.0) * iou_sim
        )
        
        return total, {
            'spatial': float(spatial_sim),
            'area': float(area_sim),
            'shape': float(shape_sim),
            'iou': float(iou_sim),
            'total': float(total)
        }
    
    def _compute_pair_metrics(
        self,
        prev_attrs: Dict[str, Any],
        curr_attrs: Dict[str, Any],
        max_dist: float
    ) -> Dict[str, float]:
        """Compute comprehensive metrics for a crown pair."""
        prev_geom = prev_attrs['geometry']
        curr_geom = curr_attrs['geometry']
        
        try:
            intersection_area = prev_geom.intersection(curr_geom).area
        except Exception:
            intersection_area = 0.0
        
        try:
            union_area = prev_geom.union(curr_geom).area
        except Exception:
            union_area = prev_attrs['area'] + curr_attrs['area'] - intersection_area
        
        prev_area = prev_attrs['area'] if prev_attrs['area'] > 0 else 1e-6
        curr_area = curr_attrs['area'] if curr_attrs['area'] > 0 else 1e-6
        
        overlap_prev = intersection_area / prev_area
        overlap_curr = intersection_area / curr_area
        iou = intersection_area / union_area if union_area > 0 else 0.0
        
        centroid_dist = prev_attrs['centroid'].distance(curr_attrs['centroid'])
        base_similarity, parts = self._weighted_similarity(prev_attrs, curr_attrs, max_dist=max_dist)
        
        prev_radius = np.sqrt(prev_area / np.pi)
        curr_radius = np.sqrt(curr_area / np.pi)
        mean_radius = max((prev_radius + curr_radius) / 2.0, 1e-3)
        
        area_ratio = curr_area / prev_area if prev_area > 0 else np.inf
        if not np.isfinite(area_ratio) or area_ratio <= 0:
            balanced_area_ratio = 0.0
        else:
            balanced_area_ratio = area_ratio if area_ratio <= 1 else 1 / area_ratio
        
        try:
            prev_contains_curr = prev_geom.buffer(0).contains(curr_geom)
        except Exception:
            prev_contains_curr = False
        
        try:
            curr_contains_prev = curr_geom.buffer(0).contains(prev_geom)
        except Exception:
            curr_contains_prev = False
        
        return {
            'intersection_area': float(intersection_area),
            'union_area': float(union_area),
            'overlap_prev': float(overlap_prev),
            'overlap_curr': float(overlap_curr),
            'iou': float(iou),
            'centroid_dist': float(centroid_dist),
            'base_similarity': float(base_similarity),
            'spatial_similarity': float(parts['spatial']),
            'area_similarity': float(parts['area']),
            'shape_similarity': float(parts['shape']),
            'mean_radius': float(mean_radius),
            'area_ratio': float(area_ratio if np.isfinite(area_ratio) else 0.0),
            'balanced_area_ratio': float(balanced_area_ratio),
            'prev_contains_curr': bool(prev_contains_curr),
            'curr_contains_prev': bool(curr_contains_prev),
        }
    
    def _classify_match_case(
        self,
        prev_node: Tuple[int, int],
        curr_node: Tuple[int, int],
        features: Dict[str, float],
        prev_overlap_counts: Dict[Tuple[int, int], int],
        curr_overlap_counts: Dict[Tuple[int, int], int],
        overlap_gate: float,
        mode: str = "balanced"
    ) -> str:
        """Classify the type of match between two crowns."""
        if features['prev_contains_curr'] or features['curr_contains_prev']:
            return 'containment'
        
        overlap_prev = features['overlap_prev']
        overlap_curr = features['overlap_curr']
        iou = features['iou']
        centroid_dist = features['centroid_dist']
        prev_count = prev_overlap_counts.get(prev_node, 0)
        curr_count = curr_overlap_counts.get(curr_node, 0)
        
        if mode == "lite":
            min_overlap_one_to_one = max(overlap_gate, 0.10)
            min_iou_one_to_one = max(overlap_gate * 0.5, 0.04)
            if (overlap_prev >= min_overlap_one_to_one and
                overlap_curr >= min_overlap_one_to_one and
                iou >= min_iou_one_to_one and
                centroid_dist < 50.0):
                return 'one_to_one'
            near_gate = max(overlap_gate * 0.5, 0.01)
            if centroid_dist < 50.0 and (overlap_prev >= near_gate or overlap_curr >= near_gate):
                return 'nearby'
        elif mode == "strict":
            min_overlap_for_one_to_one = max(overlap_gate, 0.30)
            min_iou_for_one_to_one = max(overlap_gate / 2, 0.10)
            if (prev_count == 1 and curr_count == 1 and 
                overlap_prev >= min_overlap_for_one_to_one and 
                overlap_curr >= min_overlap_for_one_to_one and 
                iou >= min_iou_for_one_to_one):
                return 'one_to_one'
            if centroid_dist < 30.0 and (overlap_prev >= overlap_gate or overlap_curr >= overlap_gate):
                return 'nearby'
        else:
            min_overlap_one_to_one = max(overlap_gate, 0.15)
            min_iou_one_to_one = max(overlap_gate * 0.5, 0.05)
            if (overlap_prev >= min_overlap_one_to_one and
                overlap_curr >= min_overlap_one_to_one and
                iou >= min_iou_one_to_one and
                centroid_dist < 40.0):
                return 'one_to_one'
            near_gate = max(overlap_gate * 0.5, 0.02)
            if centroid_dist < 35.0 and (overlap_prev >= near_gate or overlap_curr >= near_gate):
                return 'nearby'
        
        return 'none'
    
    def _score_candidate(
        self,
        base_similarity: float,
        similarity_parts: Dict[str, float],
        features: Dict[str, float],
        config: MatchCaseConfig
    ) -> float:
        """Score a candidate match using weighted components."""
        centroid_factor = 1.0 - min(1.0, features['centroid_dist'] / (features['mean_radius'] * 3.0))
        
        components = {
            'base': base_similarity,
            'spatial': similarity_parts.get('spatial', 0.0),
            'area': similarity_parts.get('area', 0.0),
            'shape': similarity_parts.get('shape', 0.0),
            'iou': features['iou'],
            'overlap_prev': features['overlap_prev'],
            'overlap_curr': features['overlap_curr'],
            'centroid': max(0.0, centroid_factor),
            'area_balance': features.get('balanced_area_ratio', 0.0),
        }
        
        score = 0.0
        for key, weight in config.scoring_weights.items():
            score += weight * components.get(key, 0.0)
        
        return score
    
    def _select_candidates_by_case(
        self,
        candidates: List[Dict[str, Any]],
        configs: Dict[str, MatchCaseConfig],
        case_order: List[str],
        max_dist: float,
        min_base_similarity: float = 0.0
    ) -> List[Dict[str, Any]]:
        """Select best candidates for each case in priority order."""
        selected: List[Dict[str, Any]] = []
        used_prev: Dict[Tuple[int, int], int] = defaultdict(int)
        used_curr: Dict[Tuple[int, int], int] = defaultdict(int)
        
        for case_name in case_order:
            config = configs.get(case_name)
            if not config:
                continue
            
            case_candidates = [cand for cand in candidates if cand['case'] == case_name]
            if not case_candidates:
                continue
            
            # Enrich candidates with scores
            enriched: List[Dict[str, Any]] = []
            for cand in case_candidates:
                prev_attrs = cand['prev_attrs']
                curr_attrs = cand['curr_attrs']
                features = cand['features']
                
                # Apply filters
                if config.max_centroid_dist is not None and features['centroid_dist'] > config.max_centroid_dist:
                    continue
                if features['iou'] < config.min_iou:
                    continue
                if features['overlap_prev'] < config.min_overlap_prev:
                    continue
                if features['overlap_curr'] < config.min_overlap_curr:
                    continue
                
                # Compute score (case-specific weights)
                base_similarity, parts = self._weighted_similarity(
                    prev_attrs, curr_attrs,
                    weights=config.base_similarity_weights,
                    max_dist=max_dist
                )
                if min_base_similarity and base_similarity < min_base_similarity:
                    continue
                score = self._score_candidate(base_similarity, parts, features, config)
                
                if score < config.similarity_threshold:
                    continue
                
                cand['base_similarity'] = float(base_similarity)
                cand['similarity_parts'] = {k: float(v) for k, v in parts.items()}
                cand['score'] = float(score)
                enriched.append(cand)
            
            if not enriched:
                continue
            
            # Select candidates (allow multiple edges when configured)
            if config.mutual_best:
                best_prev: Dict[Tuple[int, int], Dict[str, Any]] = {}
                best_curr: Dict[Tuple[int, int], Dict[str, Any]] = {}
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if not config.allow_multiple:
                        if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                            continue
                    
                    if prev_node not in best_prev or cand['score'] > best_prev[prev_node]['score']:
                        best_prev[prev_node] = cand
                    if curr_node not in best_curr or cand['score'] > best_curr[curr_node]['score']:
                        best_curr[curr_node] = cand
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if best_prev.get(prev_node) is cand and best_curr.get(curr_node) is cand:
                        if not config.allow_multiple:
                            if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                                continue
                        
                        if config.max_edges_per_prev is not None and used_prev[prev_node] >= config.max_edges_per_prev:
                            continue
                        if config.max_edges_per_curr is not None and used_curr[curr_node] >= config.max_edges_per_curr:
                            continue
                        
                        selected.append(cand)
                        used_prev[prev_node] += 1
                        used_curr[curr_node] += 1
            else:
                enriched.sort(key=lambda c: c['score'], reverse=True)
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if not config.allow_multiple:
                        if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                            continue
                    
                    if config.max_edges_per_prev is not None and used_prev[prev_node] >= config.max_edges_per_prev:
                        continue
                    if config.max_edges_per_curr is not None and used_curr[curr_node] >= config.max_edges_per_curr:
                        continue
                    
                    selected.append(cand)
                    used_prev[prev_node] += 1
                    used_curr[curr_node] += 1
        
        return selected
    
    def reset_graph(self) -> None:
        """Clear the graph."""
        self.G = nx.DiGraph()
    
    def build_graph_conditional(
        self,
        case_configs: Optional[Dict[str, MatchCaseConfig]] = None,
        case_order: Optional[List[str]] = None,
        base_max_dist: float = 200.0,
        overlap_gate: float = 0.05,
        min_base_similarity: float = 0.0,
        max_candidates_per_prev: Optional[int] = None,
        max_candidates_per_curr: Optional[int] = None,
        classify_mode: Optional[str] = None
    ) -> None:
        """
        Build tracking graph using conditional case-based matching.
        
        Args:
            case_configs: Dictionary of case configurations (default: ultra-relaxed)
            case_order: Priority order for cases (default: ['one_to_one', 'containment', 'nearby'])
            base_max_dist: Maximum centroid distance for candidate matches (meters)
            overlap_gate: Minimum overlap fraction to count as overlapping
            min_base_similarity: Minimum base similarity (case-specific weights) for candidate filtering
            max_candidates_per_prev: Maximum candidates to consider per previous crown
            max_candidates_per_curr: Maximum candidates to consider per current crown
            classify_mode: one of {'balanced','lite','strict'} (default: self.match_case_mode)
        """
        if not self.crowns_gdfs:
            self.load_data(load_images=False)
        
        self.reset_graph()
        
        configs = {name: replace(cfg) for name, cfg in (case_configs or self.case_configs).items()}
        order = case_order or self.case_order
        mode = classify_mode or self.match_case_mode
        
        self.last_case_counts = {}
        self.last_selected_counts = {name: 0 for name in configs.keys()}
        
        print(f"\n{'='*70}")
        print("BUILDING TRACKING GRAPH")
        print(f"{'='*70}")
        print(f"Parameters:")
        print(f"  base_max_dist: {base_max_dist}m")
        print(f"  overlap_gate: {overlap_gate}")
        print(f"  min_base_similarity: {min_base_similarity}")
        print(f"  classify_mode: {mode}")
        print(f"  case_order: {', '.join(order)}")
        print()
        
        # Add all nodes
        for idx in range(len(self.om_ids)):
            om_id = self.om_ids[idx]
            gdf = self.crowns_gdfs[om_id]
            
            for crown_id, row in gdf.iterrows():
                attrs = self.crown_attrs[om_id][crown_id]
                self.G.add_node((om_id, crown_id), **attrs)
            
            if idx == 0:
                continue
            
            # Build edges between consecutive OMs
            prev_om = self.om_ids[idx - 1]
            prev_nodes = [(prev_om, i) for i in range(len(self.crowns_gdfs[prev_om]))]
            curr_nodes = [(om_id, j) for j in range(len(gdf))]
            
            print(f"OM{prev_om} → OM{om_id}: {len(prev_nodes)} × {len(curr_nodes)} potential pairs")
            
            # Generate candidates
            candidates: List[Dict[str, Any]] = []
            overlap_counts_prev: Dict[Tuple[int, int], int] = defaultdict(int)
            overlap_counts_curr: Dict[Tuple[int, int], int] = defaultdict(int)
            
            for prev_node in prev_nodes:
                prev_attrs = self.G.nodes[prev_node]
                
                for curr_node in curr_nodes:
                    curr_attrs = self.crown_attrs[om_id][curr_node[1]]
                    
                    features = self._compute_pair_metrics(prev_attrs, curr_attrs, max_dist=base_max_dist)
                    
                    # Early filtering (distance only)
                    if features['centroid_dist'] > base_max_dist:
                        continue
                    
                    cand = {
                        'prev_node': prev_node,
                        'curr_node': curr_node,
                        'prev_attrs': prev_attrs,
                        'curr_attrs': curr_attrs,
                        'features': features,
                    }
                    candidates.append(cand)
                    
                    if features['overlap_prev'] >= overlap_gate:
                        overlap_counts_prev[prev_node] += 1
                    if features['overlap_curr'] >= overlap_gate:
                        overlap_counts_curr[curr_node] += 1
            
            if not candidates:
                print(f"  No candidates found")
                continue
            
            # Classify candidates
            for cand in candidates:
                cand['case'] = self._classify_match_case(
                    cand['prev_node'], cand['curr_node'], cand['features'],
                    overlap_counts_prev, overlap_counts_curr, overlap_gate, mode=mode
                )
            
            candidates = [cand for cand in candidates if cand['case'] != 'none']
            
            if not candidates:
                print(f"  No valid cases found")
                continue
            
            # Trim candidates if requested
            if max_candidates_per_prev is not None:
                grouped_prev: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in candidates:
                    grouped_prev[cand['prev_node']].append(cand)
                
                trimmed: List[Dict[str, Any]] = []
                for group in grouped_prev.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed.extend(group[:max_candidates_per_prev])
                candidates = trimmed
            
            if max_candidates_per_curr is not None:
                grouped_curr: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in candidates:
                    grouped_curr[cand['curr_node']].append(cand)
                
                trimmed_curr: List[Dict[str, Any]] = []
                for group in grouped_curr.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed_curr.extend(group[:max_candidates_per_curr])
                candidates = trimmed_curr
            
            # Count candidates by case
            case_counts = defaultdict(int)
            for cand in candidates:
                case_counts[cand['case']] += 1
            
            for case_name, count in case_counts.items():
                self.last_case_counts[case_name] = self.last_case_counts.get(case_name, 0) + count
            
            print(f"  Candidates by case: {dict(case_counts)}")
            
            # Select best candidates
            selected = self._select_candidates_by_case(
                candidates, configs, order, base_max_dist, min_base_similarity=min_base_similarity
            )
            
            # Add edges to graph
            for cand in selected:
                case_name = cand['case']
                features = cand['features']
                similarity_parts = cand.get('similarity_parts', {})
                
                self.G.add_edge(
                    cand['prev_node'],
                    cand['curr_node'],
                    similarity=float(cand.get('score', features['base_similarity'])),
                    method='conditional',
                    case=case_name,
                    overlap_prev=float(features['overlap_prev']),
                    overlap_curr=float(features['overlap_curr']),
                    iou=float(features['iou']),
                    centroid_distance=float(features['centroid_dist']),
                    base_similarity=float(cand.get('base_similarity', features['base_similarity'])),
                    spatial_similarity=float(similarity_parts.get('spatial', features['spatial_similarity'])),
                    area_similarity=float(similarity_parts.get('area', features['area_similarity'])),
                    shape_similarity=float(similarity_parts.get('shape', features['shape_similarity']))
                )
                
                self.last_selected_counts[case_name] = self.last_selected_counts.get(case_name, 0) + 1
            
            print(f"  Selected: {len(selected)} edges")
            print()
        
        print(f"{'='*70}")
        print(f"Graph built: {self.G.number_of_nodes()} nodes, {self.G.number_of_edges()} edges")
        print(f"{'='*70}\n")
    
    def debug_candidate_flow(
        self,
        base_max_dist: float,
        overlap_gate: float,
        min_base_similarity: float = 0.0,
        case_configs: Optional[Dict[str, MatchCaseConfig]] = None,
        case_order: Optional[List[str]] = None,
        classify_mode: Optional[str] = None,
        max_candidates_per_prev: Optional[int] = None,
        max_candidates_per_curr: Optional[int] = None
    ) -> Dict[str, Any]:
        """
        Diagnose candidate filtering stages and report potential early-discard risks.
        """
        if not self.crowns_gdfs:
            raise ValueError("No crown data loaded. Call load_data() first.")
        configs = {name: replace(cfg) for name, cfg in (case_configs or self.case_configs).items()}
        order = case_order or self.case_order
        mode = classify_mode or self.match_case_mode
        
        summary: Dict[str, Any] = {
            'pairs': {},
            'legacy_prefilter_discards': 0,
            'legacy_prefilter_discards_that_would_pass': 0
        }
        
        for idx in range(1, len(self.om_ids)):
            prev_om = self.om_ids[idx - 1]
            curr_om = self.om_ids[idx]
            prev_nodes = [(prev_om, i) for i in range(len(self.crowns_gdfs[prev_om]))]
            curr_nodes = [(curr_om, j) for j in range(len(self.crowns_gdfs[curr_om]))]
            total_pairs = len(prev_nodes) * len(curr_nodes)
            
            candidates = []
            overlap_counts_prev: Dict[Tuple[int, int], int] = defaultdict(int)
            overlap_counts_curr: Dict[Tuple[int, int], int] = defaultdict(int)
            
            dist_pass = 0
            for prev_node in prev_nodes:
                prev_attrs = self.crown_attrs[prev_om][prev_node[1]]
                for curr_node in curr_nodes:
                    curr_attrs = self.crown_attrs[curr_om][curr_node[1]]
                    features = self._compute_pair_metrics(prev_attrs, curr_attrs, max_dist=base_max_dist)
                    if features['centroid_dist'] > base_max_dist:
                        continue
                    dist_pass += 1
                    cand = {
                        'prev_node': prev_node,
                        'curr_node': curr_node,
                        'prev_attrs': prev_attrs,
                        'curr_attrs': curr_attrs,
                        'features': features,
                    }
                    candidates.append(cand)
                    if features['overlap_prev'] >= overlap_gate:
                        overlap_counts_prev[prev_node] += 1
                    if features['overlap_curr'] >= overlap_gate:
                        overlap_counts_curr[curr_node] += 1
            
            for cand in candidates:
                cand['case'] = self._classify_match_case(
                    cand['prev_node'], cand['curr_node'], cand['features'],
                    overlap_counts_prev, overlap_counts_curr, overlap_gate, mode=mode
                )
            classified = [c for c in candidates if c['case'] != 'none']
            
            # Trim candidates if requested
            if max_candidates_per_prev is not None:
                grouped_prev: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in classified:
                    grouped_prev[cand['prev_node']].append(cand)
                trimmed: List[Dict[str, Any]] = []
                for group in grouped_prev.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed.extend(group[:max_candidates_per_prev])
                classified = trimmed
            
            if max_candidates_per_curr is not None:
                grouped_curr: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in classified:
                    grouped_curr[cand['curr_node']].append(cand)
                trimmed_curr: List[Dict[str, Any]] = []
                for group in grouped_curr.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed_curr.extend(group[:max_candidates_per_curr])
                classified = trimmed_curr
            
            # Score with case-specific weights
            scored = 0
            for cand in classified:
                cfg = configs.get(cand['case'])
                if not cfg:
                    continue
                features = cand['features']
                if cfg.max_centroid_dist is not None and features['centroid_dist'] > cfg.max_centroid_dist:
                    continue
                if features['iou'] < cfg.min_iou:
                    continue
                if features['overlap_prev'] < cfg.min_overlap_prev:
                    continue
                if features['overlap_curr'] < cfg.min_overlap_curr:
                    continue
                base_similarity, parts = self._weighted_similarity(
                    cand['prev_attrs'], cand['curr_attrs'],
                    weights=cfg.base_similarity_weights, max_dist=base_max_dist
                )
                if min_base_similarity and base_similarity < min_base_similarity:
                    continue
                score = self._score_candidate(base_similarity, parts, features, cfg)
                if score < cfg.similarity_threshold:
                    continue
                scored += 1
                
                # legacy prefilter test (old behavior)
                legacy_reject = features['base_similarity'] < min_base_similarity and features['iou'] < overlap_gate
                if legacy_reject:
                    summary['legacy_prefilter_discards'] += 1
                    summary['legacy_prefilter_discards_that_would_pass'] += 1
            
            summary['pairs'][f"{prev_om}->{curr_om}"] = {
                'total_pairs': total_pairs,
                'within_distance': dist_pass,
                'classified_candidates': len(classified),
                'case_scored_pass': scored,
            }
        
        return summary
    
    def _extract_all_chains(self) -> List[List[Tuple[int, int]]]:
        """Extract all tracking chains from the graph."""
        visited = set()
        chains: List[List[Tuple[int, int]]] = []
        
        # Start from nodes with no predecessors
        chain_starts = [n for n in self.G.nodes if not list(self.G.predecessors(n))]
        
        for start_node in chain_starts:
            if start_node in visited:
                continue
            
            chain = self._greedy_chain(start_node)
            chains.append(chain)
            visited.update(chain)
        
        # Handle remaining nodes (cycles or isolated)
        remaining = set(self.G.nodes) - visited
        for node in remaining:
            chains.append([node])
        
        return chains
    
    def _greedy_chain(self, start_node: Tuple[int, int]) -> List[Tuple[int, int]]:
        """Extract a single chain starting from a given node."""
        chain = [start_node]
        current = start_node
        
        while True:
            successors = list(self.G.successors(current))
            if not successors:
                break
            
            if len(successors) > 1:
                # Choose best successor
                best_successor = max(
                    successors,
                    key=lambda n: self.G[current][n].get('similarity', 0.0)
                )
                chain.append(best_successor)
                current = best_successor
            else:
                chain.append(successors[0])
                current = successors[0]
        
        return chain
    
    def quality_report(self) -> Tuple[str, Dict[str, Any]]:
        """Generate quality metrics and report."""
        G = self.G
        om_ids = self.om_ids
        
        metrics: Dict[str, Any] = {
            'total_trees_detected': G.number_of_nodes(),
            'total_edges': G.number_of_edges(),
            'total_possible_matches': 0,
            'successful_matches': 0,
            'match_rate_by_om_pair': {},
            'chain_length_distribution': {},
            'average_chain_length': 0,
            'median_chain_length': 0,
            'max_chain_length': 0,
        }
        
        # Chain analysis
        chains = self._extract_all_chains()
        chain_lengths = [len(chain) for chain in chains]
        
        if chain_lengths:
            metrics['average_chain_length'] = float(np.mean(chain_lengths))
            metrics['median_chain_length'] = float(np.median(chain_lengths))
            metrics['max_chain_length'] = int(max(chain_lengths))
            
            for length in chain_lengths:
                metrics['chain_length_distribution'][int(length)] = metrics['chain_length_distribution'].get(int(length), 0) + 1
        
        # Match rate by OM pair
        for i in range(len(om_ids) - 1):
            om1, om2 = om_ids[i], om_ids[i + 1]
            om1_nodes = [n for n in G.nodes if n[0] == om1]
            om2_nodes = [n for n in G.nodes if n[0] == om2]
            
            matches = sum(1 for u, v in G.edges() if u[0] == om1 and v[0] == om2)
            possible_matches = min(len(om1_nodes), len(om2_nodes))
            match_rate = matches / possible_matches if possible_matches > 0 else 0.0
            
            metrics['match_rate_by_om_pair'][f"{om1}->{om2}"] = {
                'matches': matches,
                'possible': possible_matches,
                'rate': float(match_rate),
            }
            
            metrics['total_possible_matches'] += possible_matches
            metrics['successful_matches'] += matches
        
        metrics['overall_match_rate'] = (
            metrics['successful_matches'] / metrics['total_possible_matches']
            if metrics['total_possible_matches'] > 0 else 0.0
        )
        
        # Generate report
        report = [
            "# Tree Tracking Quality Assessment Report",
            f"Total Trees Detected: {metrics['total_trees_detected']}",
            f"Total Tracking Edges: {metrics['total_edges']}",
            f"Overall Match Rate: {metrics['overall_match_rate']:.3f}",
            f"Average Chain Length: {metrics.get('average_chain_length', 0):.2f}",
            f"Maximum Chain Length: {metrics.get('max_chain_length', 0)}",
            "\nMatch Rates by Orthomosaic Pair:",
        ]
        
        for pair, data in metrics['match_rate_by_om_pair'].items():
            report.append(f"- {pair}: {data['matches']}/{data['possible']} ({data['rate']:.3f})")
        
        report.append("\nChain Length Distribution:")
        for length, count in sorted(metrics['chain_length_distribution'].items()):
            report.append(f"- Length {length}: {count} trees")
        
        if self.last_selected_counts:
            report.append("\nEdge selection by case:")
            for case_name, count in sorted(self.last_selected_counts.items(), key=lambda kv: (-kv[1], kv[0])):
                total_candidates = self.last_case_counts.get(case_name, 0)
                if total_candidates:
                    ratio = count / total_candidates
                    report.append(f"- {case_name}: {count} / {total_candidates} ({ratio:.2f})")
                else:
                    report.append(f"- {case_name}: {count}")
        
        return "\n".join(report), metrics
    
    def get_matching_chain(self, start_om_id: int, crown_id: int) -> List[Tuple[int, int]]:
        """Get the tracking chain starting from a specific crown."""
        node = (start_om_id, crown_id)
        if node not in self.G:
            raise ValueError(f"Node {(start_om_id, crown_id)} not in graph. Build the graph first.")
        return self._greedy_chain(node)
    
    def ensure_output_dir(self) -> None:
        """Create output directory if it doesn't exist."""
        os.makedirs(self.output_dir, exist_ok=True)
    
    def save_text(self, text: str, filename: str) -> str:
        """Save text to file in output directory."""
        self.ensure_output_dir()
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w') as f:
            f.write(text)
        return path
    
    def save_json(self, data: Dict[str, Any], filename: str) -> str:
        """Save JSON data to file in output directory."""
        self.ensure_output_dir()
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        return path

print("✓ TreeTrackingGraph class defined with configurable alignment and match modes")

✓ TreeTrackingGraph class defined with configurable alignment and match modes


In [3]:
# ---- Phase-correlation alignment (orthomosaics) ----
def _read_ortho_lowres(path: str, max_size: int = 1200):
    with rasterio.open(path) as src:
        scale = max(src.width, src.height) / max_size if max(src.width, src.height) > max_size else 1.0
        out_w = int(round(src.width / scale))
        out_h = int(round(src.height / scale))
        data = src.read([1, 2, 3], out_shape=(3, out_h, out_w), resampling=rasterio.enums.Resampling.bilinear)
        rgb = np.moveaxis(data, 0, -1).astype(np.float32)
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
        gray = 0.2989 * rgb[..., 0] + 0.5870 * rgb[..., 1] + 0.1140 * rgb[..., 2]
        scale_x = src.width / out_w
        scale_y = src.height / out_h
        transform = src.transform * rasterio.transform.Affine.scale(scale_x, scale_y)
        return rgb, gray, transform

def _bounds_from_transform(h: int, w: int, transform: rasterio.transform.Affine):
    corners = [(0, 0), (0, w), (h, 0), (h, w)]
    xs = []
    ys = []
    for row, col in corners:
        x, y = rasterio.transform.xy(transform, row, col, offset='ul')
        xs.append(x)
        ys.append(y)
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    return left, bottom, right, top

def _safe_window(win: rasterio.windows.Window, height: int, width: int) -> rasterio.windows.Window:
    col_off = int(max(0, min(width, win.col_off)))
    row_off = int(max(0, min(height, win.row_off)))
    max_w = max(0, width - col_off)
    max_h = max(0, height - row_off)
    w = int(max(0, min(max_w, win.width)))
    h = int(max(0, min(max_h, win.height)))
    return rasterio.windows.Window(col_off=col_off, row_off=row_off, width=w, height=h)

def _crop_to_overlap(ref_img, mov_img, ref_transform, mov_transform):
    ref_h, ref_w = ref_img.shape[:2]
    mov_h, mov_w = mov_img.shape[:2]
    ref_left, ref_bottom, ref_right, ref_top = _bounds_from_transform(ref_h, ref_w, ref_transform)
    mov_left, mov_bottom, mov_right, mov_top = _bounds_from_transform(mov_h, mov_w, mov_transform)
    left = max(ref_left, mov_left)
    right = min(ref_right, mov_right)
    bottom = max(ref_bottom, mov_bottom)
    top = min(ref_top, mov_top)
    if left >= right or bottom >= top:
        return None, None
    ref_win = rasterio.windows.from_bounds(left, bottom, right, top, ref_transform).round_offsets().round_lengths()
    mov_win = rasterio.windows.from_bounds(left, bottom, right, top, mov_transform).round_offsets().round_lengths()
    ref_win = _safe_window(ref_win, ref_h, ref_w)
    mov_win = _safe_window(mov_win, mov_h, mov_w)
    if ref_win.width < 2 or ref_win.height < 2 or mov_win.width < 2 or mov_win.height < 2:
        return None, None
    ref_crop = ref_img[int(ref_win.row_off): int(ref_win.row_off + ref_win.height),
                       int(ref_win.col_off): int(ref_win.col_off + ref_win.width)]
    mov_crop = mov_img[int(mov_win.row_off): int(mov_win.row_off + mov_win.height),
                       int(mov_win.col_off): int(mov_win.col_off + mov_win.width)]
    return ref_crop, mov_crop

def _match_shape(ref_crop, mov_crop):
    if ref_crop.shape == mov_crop.shape:
        return ref_crop, mov_crop
    if not SKIMAGE_AVAILABLE:
        min_h = min(ref_crop.shape[0], mov_crop.shape[0])
        min_w = min(ref_crop.shape[1], mov_crop.shape[1])
        return ref_crop[:min_h, :min_w], mov_crop[:min_h, :min_w]
    mov_resized = resize(mov_crop, ref_crop.shape, preserve_range=True, anti_aliasing=True)
    return ref_crop, mov_resized

def _phase_corr_shift(ref_gray, mov_gray, ref_transform, mov_transform):
    ref_crop, mov_crop = _crop_to_overlap(ref_gray, mov_gray, ref_transform, mov_transform)
    if ref_crop is None or mov_crop is None:
        return None, None
    ref_crop, mov_crop = _match_shape(ref_crop, mov_crop)
    shift, error, _ = phase_cross_correlation(ref_crop, mov_crop, upsample_factor=10)
    shift_row, shift_col = shift
    dx = shift_col * ref_transform.a
    dy = shift_row * ref_transform.e
    return (float(dx), float(dy)), {'shift_px': (float(shift_row), float(shift_col)), 'error': float(error)}

def phase_corr_align_tracker(tracker: 'TreeTrackingGraph', reference_om_id: Optional[int] = None, max_preview: int = 1200):
    if not SKIMAGE_AVAILABLE:
        raise RuntimeError('scikit-image not available; cannot run phase correlation alignment.')
    if reference_om_id is None:
        reference_om_id = tracker.om_ids[0]
    # Load low-res orthos
    ortho_gray = {}
    ortho_transform = {}
    for om_id, (_, ortho_file) in zip(tracker.om_ids, tracker.file_pairs):
        if ortho_file and os.path.exists(ortho_file):
            _, gray, transform = _read_ortho_lowres(ortho_file, max_preview)
            ortho_gray[om_id] = gray
            ortho_transform[om_id] = transform
        else:
            ortho_gray[om_id] = None
            ortho_transform[om_id] = None
    shifts = {reference_om_id: (0.0, 0.0)}
    # Build cumulative shifts across consecutive OMs
    for idx in range(1, len(tracker.om_ids)):
        prev_id = tracker.om_ids[idx - 1]
        curr_id = tracker.om_ids[idx]
        ref_gray = ortho_gray.get(prev_id)
        mov_gray = ortho_gray.get(curr_id)
        ref_transform = ortho_transform.get(prev_id)
        mov_transform = ortho_transform.get(curr_id)
        if ref_gray is None or mov_gray is None or ref_transform is None or mov_transform is None:
            shifts[curr_id] = shifts.get(prev_id, (0.0, 0.0))
            continue
        out = _phase_corr_shift(ref_gray, mov_gray, ref_transform, mov_transform)
        if out[0] is None:
            shifts[curr_id] = shifts.get(prev_id, (0.0, 0.0))
            continue
        (dx, dy), meta = out
        prev_dx, prev_dy = shifts.get(prev_id, (0.0, 0.0))
        shifts[curr_id] = (prev_dx + dx, prev_dy + dy)
    # Apply shifts
    for om_id in tracker.om_ids:
        if om_id == reference_om_id:
            continue
        dx, dy = shifts.get(om_id, (0.0, 0.0))
        gdf = tracker.crowns_gdfs[om_id].copy()
        gdf['geometry'] = gdf['geometry'].apply(lambda g: translate(g, xoff=dx, yoff=dy))
        tracker.crowns_gdfs[om_id] = gdf
        tracker.crown_attrs[om_id] = [tracker._compute_crown_attributes(row.geometry) for _, row in gdf.iterrows()]
    tracker.alignment_shifts = shifts
    return shifts

## 3. Visualization Functions

Functions for visualizing tracking chains with polygons and extracted RGB images.

In [4]:
def visualize_chain_with_extracted_images(chain, aligned_gdfs, crown_images_dict, tracker, title="Chain Example", save_path=None, show=True, close_fig=True, dpi=150):
    """
    Visualize chain with both crown polygons AND extracted RGB images.
    Optional saving for PPT-friendly reuse.

    Args:
        chain: List of (om_id, crown_idx) tuples
        aligned_gdfs: Dictionary of aligned GeoDataFrames by OM ID
        crown_images_dict: Dictionary of crown images by OM ID (from tracker.crown_images)
        tracker: TreeTrackingGraph instance
        title: Plot title
        save_path: Optional path to save the figure
        show: Whether to display the plot
        close_fig: Close the figure after saving/showing to free memory
        dpi: Resolution when saving
    """
    chain_length = len(chain)
    
    # Create figure with 2 rows: polygons on top, images on bottom
    fig = plt.figure(figsize=(5*chain_length, 10))
    
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")
    print(f"Chain: {chain}")
    print(f"Length: {chain_length}\n")
    
    # Plot each crown in the chain
    for idx, (om_idx, crown_idx) in enumerate(chain):
        gdf = aligned_gdfs[om_idx]
        crown = gdf.iloc[crown_idx]
        
        # Get edge info if not last crown
        edge_info = ""
        if idx < chain_length - 1:
            next_node = chain[idx + 1]
            if tracker.G.has_edge((om_idx, crown_idx), next_node):
                edge_data = tracker.G.edges[(om_idx, crown_idx), next_node]
                edge_info = f" → OM{next_node[0]}[{next_node[1]}]: case={edge_data['case']}, sim={edge_data['similarity']:.2f}, iou={edge_data.get('iou', 0):.2f}"
        
        # Print crown info
        in_deg = tracker.G.in_degree((om_idx, crown_idx))
        out_deg = tracker.G.out_degree((om_idx, crown_idx))
        print(f"  OM{om_idx}[{crown_idx}]: in_deg={in_deg}, out_deg={out_deg}, area={crown.geometry.area:.1f}m²{edge_info}")
        
        # ROW 1: Crown polygon
        ax_poly = plt.subplot(2, chain_length, idx + 1)
        
        # Get bounds and plot
        minx, miny, maxx, maxy = crown.geometry.bounds
        margin = max((maxx - minx), (maxy - miny)) * 0.3
        
        # Plot other crowns in gray
        gdf.plot(ax=ax_poly, color='lightgray', edgecolor='gray', alpha=0.3)
        
        # Plot this crown highlighted
        gpd.GeoSeries([crown.geometry]).plot(
            ax=ax_poly,
            facecolor=plt.cm.tab10(om_idx-1),
            edgecolor='black',
            linewidth=2,
            alpha=0.7
        )
        
        # Plot centroid
        centroid = crown.geometry.centroid
        ax_poly.plot(centroid.x, centroid.y, 'o', color='yellow', markersize=8,
                    markeredgecolor='black', markeredgewidth=1.5)
        
        # Arrow to next crown
        if idx < chain_length - 1:
            ax_poly.annotate('', xy=(maxx + margin*0.5, (miny+maxy)/2),
                            xytext=(maxx + margin*0.2, (miny+maxy)/2),
                            arrowprops=dict(arrowstyle='->', lw=3, color='red'))
        
        ax_poly.set_xlim(minx - margin, maxx + margin)
        ax_poly.set_ylim(miny - margin, maxy + margin)
        ax_poly.set_aspect('equal')
        ax_poly.set_title(
            f"OM{om_idx} - Crown {crown_idx}\nArea: {crown.geometry.area:.1f}m²\nIn:{in_deg} Out:{out_deg}",
            fontsize=10, fontweight='bold'
        )
        ax_poly.set_xlabel("X (m)", fontsize=9)
        ax_poly.set_ylabel("Y (m)", fontsize=9)
        ax_poly.grid(True, alpha=0.3)
        
        # Add edge case label
        if edge_info:
            edge_data = tracker.G.edges[(om_idx, crown_idx), chain[idx + 1]]
            bbox_color = {
                'one_to_one': 'wheat',
                'containment': 'lightblue',
                'nearby': 'lightcoral'
            }.get(edge_data['case'], 'white')
            ax_poly.text(
                0.95, 0.95,
                f"→ {edge_data['case']}\nSim: {edge_data['similarity']:.2f}\nIoU: {edge_data.get('iou', 0):.2f}",
                transform=ax_poly.transAxes, fontsize=8,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor=bbox_color, alpha=0.8)
            )
        
        # ROW 2: Extracted RGB image
        ax_img = plt.subplot(2, chain_length, chain_length + idx + 1)
        
        rgb_image = crown_images_dict[om_idx][crown_idx]
        if rgb_image is not None:
            ax_img.imshow(rgb_image)
            ax_img.set_title(
                f"Extracted Tree Image\n{rgb_image.shape[0]}×{rgb_image.shape[1]} px",
                fontsize=9
            )
        else:
            ax_img.text(0.5, 0.5, 'Image not available',
                       ha='center', va='center', fontsize=10, color='red')
            ax_img.set_title("No Image", fontsize=9)
        
        ax_img.axis('off')
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    if show:
        plt.show()
    if close_fig:
        plt.close(fig)
    print()
    return save_path


def categorize_chains(tracker):
    """
    Categorize all chains from the tracker.
    
    Returns:
        Dictionary with categorized chains
    """
    all_chains = tracker._extract_all_chains()
    max_chain_length = len(tracker.om_ids)
    
    full_chains_width_1 = []      # Full length chains with no branching
    full_chains_branching = []    # Full length chains with branching
    partial_chains_long = []      # Length 3-4
    partial_chains_short = []     # Length 2
    
    for chain in all_chains:
        chain_length = len(chain)
        
        if chain_length == max_chain_length:
            # Full chain - check for branching
            has_branching = False
            for node in chain:
                in_degree = tracker.G.in_degree(node)
                out_degree = tracker.G.out_degree(node)
                if in_degree > 1 or out_degree > 1:
                    has_branching = True
                    break
            
            if has_branching:
                full_chains_branching.append(chain)
            else:
                full_chains_width_1.append(chain)
        else:
            # Partial chains
            if chain_length >= 3:
                partial_chains_long.append(chain)
            elif chain_length == 2:
                partial_chains_short.append(chain)
    
    return {
        'full_width_1': full_chains_width_1,
        'full_branching': full_chains_branching,
        'partial_long': partial_chains_long,
        'partial_short': partial_chains_short,
        'singleton': [c for c in all_chains if len(c) == 1]
    }


print("✓ Visualization functions loaded")

✓ Visualization functions loaded


In [5]:
# Helper to build tuned configs without monkey-patching
def make_case_configs(mode="balanced"):
    if mode == "lite":
        return {
            'one_to_one': MatchCaseConfig(
                name='one_to_one',
                base_similarity_weights={'spatial': 0.35, 'area': 0.15, 'shape': 0.15, 'iou': 0.35},
                scoring_weights={'base': 0.40, 'iou': 0.20, 'overlap_prev': 0.15, 'overlap_curr': 0.15, 'centroid': 0.10},
                similarity_threshold=0.26,
                min_iou=0.04,
                min_overlap_prev=0.12,
                min_overlap_curr=0.12,
                max_centroid_dist=55.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
            'nearby': MatchCaseConfig(
                name='nearby',
                base_similarity_weights={'spatial': 0.80, 'area': 0.10, 'shape': 0.05, 'iou': 0.05},
                scoring_weights={'base': 0.48, 'centroid': 0.30, 'iou': 0.10, 'overlap_prev': 0.06, 'overlap_curr': 0.06},
                similarity_threshold=0.12,
                min_iou=0.0,
                min_overlap_prev=0.02,
                min_overlap_curr=0.02,
                max_centroid_dist=70.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
            'containment': MatchCaseConfig(
                name='containment',
                base_similarity_weights={'spatial': 0.40, 'area': 0.20, 'shape': 0.10, 'iou': 0.30},
                scoring_weights={'base': 0.35, 'overlap_prev': 0.30, 'overlap_curr': 0.30, 'iou': 0.05},
                similarity_threshold=0.22,
                min_iou=0.0,
                min_overlap_prev=0.18,
                min_overlap_curr=0.18,
                max_centroid_dist=100.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
        }
    
    # balanced (default)
    return {
        'one_to_one': MatchCaseConfig(
            name='one_to_one',
            base_similarity_weights={'spatial': 0.35, 'area': 0.15, 'shape': 0.15, 'iou': 0.35},
            scoring_weights={'base': 0.40, 'iou': 0.20, 'overlap_prev': 0.15, 'overlap_curr': 0.15, 'centroid': 0.10},
            similarity_threshold=0.28,
            min_iou=0.04,
            min_overlap_prev=0.15,
            min_overlap_curr=0.15,
            max_centroid_dist=45.0,
            mutual_best=False,
            allow_multiple=True,
            max_edges_per_prev=2,
            max_edges_per_curr=2,
        ),
        'nearby': MatchCaseConfig(
            name='nearby',
            base_similarity_weights={'spatial': 0.80, 'area': 0.10, 'shape': 0.05, 'iou': 0.05},
            scoring_weights={'base': 0.48, 'centroid': 0.30, 'iou': 0.10, 'overlap_prev': 0.06, 'overlap_curr': 0.06},
            similarity_threshold=0.12,
            min_iou=0.0,
            min_overlap_prev=0.02,
            min_overlap_curr=0.02,
            max_centroid_dist=70.0,
            mutual_best=False,
            allow_multiple=True,
            max_edges_per_prev=2,
            max_edges_per_curr=2,
        ),
        'containment': MatchCaseConfig(
            name='containment',
            base_similarity_weights={'spatial': 0.40, 'area': 0.20, 'shape': 0.10, 'iou': 0.30},
            scoring_weights={'base': 0.35, 'overlap_prev': 0.30, 'overlap_curr': 0.30, 'iou': 0.05},
            similarity_threshold=0.22,
            min_iou=0.0,
            min_overlap_prev=0.20,
            min_overlap_curr=0.20,
            max_centroid_dist=100.0,
            mutual_best=False,
            allow_multiple=True,
            max_edges_per_prev=2,
            max_edges_per_curr=2,
        ),
    }

print("✓ Config factory loaded")

✓ Config factory loaded


In [6]:
# Choose classification mode: 'balanced', 'lite', or 'strict'
match_case_mode = "balanced"
print(f"✓ Match case mode set to: {match_case_mode}")

✓ Match case mode set to: balanced


## 4. Initialize and Load Data

Initialize the tracker and load crown data with image extraction.

In [7]:
# Initialize the tracker
tracker = TreeTrackingGraph(
    crown_dir='../../output/input_crowns',
    ortho_dir='../../input/input_om',
    output_dir='../../output',
    auto_discover=True
)

print(f"\nDiscovered {len(tracker.file_pairs)} orthomosaic pairs")
print(f"OM IDs: {tracker.om_ids}")
print("\nFile pairs:")
for om_id, (crown_file, ortho_file) in zip(tracker.om_ids, tracker.file_pairs):
    print(f"  OM{om_id}: {os.path.basename(crown_file)}, {os.path.basename(ortho_file) if ortho_file else 'N/A'}")


Discovered 7 orthomosaic pairs
OM IDs: [1, 2, 3, 4, 5, 6, 7]

File pairs:
  OM1: OM1.gpkg, sit_om1.tif
  OM2: OM2.gpkg, sit_om2.tif
  OM3: OM3.gpkg, sit_om3.tif
  OM4: OM4.gpkg, sit_om4.tif
  OM5: OM5.gpkg, sit_om5.tif
  OM6: OM6.gpkg, sit_om6.tif
  OM7: OM7.gpkg, sit_om7.tif


In [8]:
# Apply tuned configs (branching allowed) and classification mode
tracker.match_case_mode = match_case_mode
tracker.case_configs = make_case_configs(match_case_mode)
tracker.case_order = ['one_to_one', 'nearby', 'containment']
print("✓ Tracker configured for branching-aware matching")

✓ Tracker configured for branching-aware matching


In [9]:
# Load crown data once, extract RGB images, and align via phase correlation
tracker.load_data(load_images=True, align=False)
phase_corr_align_tracker(tracker, reference_om_id=tracker.om_ids[0], max_preview=1200)
print(f"\n✓ Loaded data from {len(tracker.om_ids)} OMs")
print(f"✓ Extracted images: {sum(sum(img is not None for img in imgs) for imgs in tracker.crown_images.values())} total")


LOADING DATA

OM1: Loading OM1.gpkg
  Extracting 82 crown images from sit_om1.tif...
  ✓ Extracted 82 images

OM2: Loading OM2.gpkg
  Extracting 85 crown images from sit_om2.tif...
  ✓ Extracted 85 images

OM3: Loading OM3.gpkg
  Extracting 78 crown images from sit_om3.tif...
  ✓ Extracted 78 images

OM4: Loading OM4.gpkg
  Extracting 80 crown images from sit_om4.tif...
  ✓ Extracted 80 images

OM5: Loading OM5.gpkg
  Extracting 83 crown images from sit_om5.tif...
  ✓ Extracted 83 images

OM6: Loading OM6.gpkg
  Extracting 93 crown images from sit_om6.tif...
  ✓ Extracted 93 images

OM7: Loading OM7.gpkg
  Extracting 95 crown images from sit_om7.tif...
  ✓ Extracted 95 images
CRS CONSISTENCY CHECK
------------------------------------------------------------
✓ Crown CRS consistent: EPSG:32643
------------------------------------------------------------


✓ Loaded data from 7 OMs
✓ Extracted images: 596 total


## 6. Build Tracking Graph

Build the crown tracking graph with ultra-relaxed parameters.

In [10]:
# Build the tracking graph with tuned configs (branching allowed)
tracker.case_order = ['one_to_one', 'nearby', 'containment']

tracker.build_graph_conditional(
    base_max_dist=50,
    overlap_gate=0.02,
    min_base_similarity=0.1,
    max_candidates_per_prev=3,
    max_candidates_per_curr=3,
    classify_mode=tracker.match_case_mode,
    case_configs=tracker.case_configs,
    case_order=tracker.case_order,
)

# Generate quality report
report, metrics = tracker.quality_report()
print(report)

# Save report
tracker.save_text(report, 'crown_tracking_quality_report.txt')
tracker.save_json(metrics, 'crown_tracking_metrics.json')


BUILDING TRACKING GRAPH
Parameters:
  base_max_dist: 50m
  overlap_gate: 0.02
  min_base_similarity: 0.1
  classify_mode: balanced
  case_order: one_to_one, nearby, containment

OM1 → OM2: 82 × 85 potential pairs
  Candidates by case: {'one_to_one': 63, 'nearby': 16, 'containment': 1}
  Selected: 74 edges

OM2 → OM3: 85 × 78 potential pairs
  Candidates by case: {'one_to_one': 56, 'nearby': 22, 'containment': 3}
  Selected: 73 edges

OM3 → OM4: 78 × 80 potential pairs
  Candidates by case: {'one_to_one': 53, 'nearby': 11, 'containment': 2}
  Selected: 60 edges

OM4 → OM5: 80 × 83 potential pairs
  Candidates by case: {'one_to_one': 57, 'nearby': 10}
  Selected: 62 edges

OM5 → OM6: 83 × 93 potential pairs
  Candidates by case: {'one_to_one': 53, 'nearby': 16}
  Selected: 61 edges

OM6 → OM7: 93 × 95 potential pairs
  Candidates by case: {'one_to_one': 69, 'nearby': 8, 'containment': 2}
  Selected: 73 edges

Graph built: 596 nodes, 403 edges

# Tree Tracking Quality Assessment Report
T

'../../output/crown_tracking_metrics.json'

In [11]:
# # Example: Visualize a specific tracking chain
# example_chain = tracker.get_matching_chain(start_om_id=1, crown_id=0)
# if len(example_chain) > 1:
#     visualize_chain_with_extracted_images(
#         example_chain, 
#         tracker.crowns_gdfs, 
#         tracker.crown_images, 
#         tracker, 
#         title="Example Tracking Chain"
#     )
# else:
#     print("No chain found for OM1, Crown 0")

## 7. Analyze Tracking Results

Categorize and analyze the tracking chains.

In [12]:
# Categorize chains
chain_categories = categorize_chains(tracker)

print("Chain Analysis:")
print(f"Full chains (no branching): {len(chain_categories['full_width_1'])}")
print(f"Full chains (with branching): {len(chain_categories['full_branching'])}")
print(f"Partial chains (long): {len(chain_categories['partial_long'])}")
print(f"Partial chains (short): {len(chain_categories['partial_short'])}")
print(f"Singletons: {len(chain_categories['singleton'])}")

# Show example of each category
for category, chains in chain_categories.items():
    if chains:
        print(f"\nExample {category} chain: {chains[0]}")

Chain Analysis:
Full chains (no branching): 9
Full chains (with branching): 11
Partial chains (long): 56
Partial chains (short): 51
Singletons: 163

Example full_width_1 chain: [(1, 9), (2, 13), (3, 33), (4, 45), (5, 47), (6, 1), (7, 30)]

Example full_branching chain: [(1, 10), (2, 56), (3, 61), (4, 27), (5, 49), (6, 79), (7, 88)]

Example partial_long chain: [(1, 0), (2, 60), (3, 42), (4, 38), (5, 34)]

Example partial_short chain: [(1, 19), (2, 37)]

Example singleton chain: [(1, 3)]


In [13]:
# # Visualize all 5 crown sets: aligned vs non-aligned for comparison
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches

# # Load original non-aligned crowns for comparison
# tracker_original = TreeTrackingGraph(
#     crown_dir='../../output/input_crowns',
#     ortho_dir='../../input/input_om',
#     output_dir='../../output',
#     auto_discover=True
# )
# tracker_original.load_data(load_images=False, align=False)  # No alignment

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))

# colors = ['red', 'blue', 'green', 'orange', 'purple']

# # Plot 1: Non-aligned crowns
# legend_handles = []
# for i, om_id in enumerate(tracker_original.om_ids):
#     gdf = tracker_original.crowns_gdfs[om_id]
#     gdf.plot(ax=ax1, facecolor='none', edgecolor=colors[i], linewidth=2, alpha=0.8)
#     legend_handles.append(mpatches.Patch(edgecolor=colors[i], facecolor='none', linewidth=2, label=f'OM{om_id}'))

# ax1.set_title('Crown Sets Before Spatial Alignment (Original Positions)')
# ax1.legend(handles=legend_handles)
# ax1.set_aspect('equal')
# ax1.set_xlabel('X (m)')
# ax1.set_ylabel('Y (m)')
# ax1.grid(True, alpha=0.3)

# # Plot 2: Aligned crowns
# legend_handles = []
# for i, om_id in enumerate(tracker.om_ids):
#     gdf = tracker.crowns_gdfs[om_id]
#     gdf.plot(ax=ax2, facecolor='none', edgecolor=colors[i], linewidth=2, alpha=0.8)
#     legend_handles.append(mpatches.Patch(edgecolor=colors[i], facecolor='none', linewidth=2, label=f'OM{om_id}'))

# ax2.set_title('Crown Sets After Spatial Alignment (Corrected Positions)')
# ax2.legend(handles=legend_handles)
# ax2.set_aspect('equal')
# ax2.set_xlabel('X (m)')
# ax2.set_ylabel('Y (m)')
# ax2.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# print("Alignment comparison: Left shows original positions, right shows after spatial corrections.")
# print("If alignment worked, crowns should be better aligned spatially on the right.")

In [14]:
# # Visualize all complete 5-length tracking chains
# print("Visualizing all complete 5-length tracking chains...")

# # Get all full chains (both no-branching and with-branching)
# full_chains = chain_categories['full_width_1'] + chain_categories['full_branching']

# print(f"Found {len(full_chains)} complete chains to visualize")

# for i, chain in enumerate(full_chains):
#     chain_type = "No Branching" if chain in chain_categories['full_width_1'] else "With Branching"
#     title = f"Complete Chain {i+1}/{len(full_chains)} - {chain_type}"

#     visualize_chain_with_extracted_images(
#         chain,
#         tracker.crowns_gdfs,
#         tracker.crown_images,
#         tracker,
#         title=title
#     )

# print(f"✓ Completed visualization of all {len(full_chains)} complete tracking chains")

## 9. Consensus Crown Strategies
Define robust strategies to build a single consensus crown polygon for a tracked chain, then visualize consensus extractions across OMs.

In [15]:
from shapely.ops import unary_union
from functools import reduce

def _chain_polygons_aligned(chain, tracker):
    polys = []
    for om_id, crown_idx in chain:
        poly = tracker.crowns_gdfs[om_id].iloc[crown_idx].geometry
        if poly is not None and not poly.is_empty:
            polys.append(poly.buffer(0))
    return polys

def _largest_polygon(geom):
    if geom is None or geom.is_empty:
        return geom
    if geom.geom_type == "Polygon":
        return geom
    if geom.geom_type in ("MultiPolygon", "GeometryCollection"):
        polys = [g for g in geom.geoms if g.geom_type == "Polygon"]
        return max(polys, key=lambda g: g.area) if polys else geom
    return geom

def consensus_medoid(chain, tracker, w_centroid=0.5, w_iou=0.4, w_area=0.1):
    polys = _chain_polygons_aligned(chain, tracker)
    if not polys:
        return None
    centroids = [p.centroid for p in polys]
    areas = [p.area for p in polys]
    max_dist = max((c1.distance(c2) for c1 in centroids for c2 in centroids), default=1.0) or 1.0
    best_idx = 0
    best_score = float("inf")
    for i, p in enumerate(polys):
        score = 0.0
        for j, q in enumerate(polys):
            if i == j:
                continue
            dist = centroids[i].distance(centroids[j]) / max_dist
            iou = tracker._compute_iou(p, q)
            area_ratio = min(areas[i], areas[j]) / max(areas[i], areas[j]) if max(areas[i], areas[j]) > 0 else 0.0
            score += (w_centroid * dist) + (w_iou * (1.0 - iou)) + (w_area * (1.0 - area_ratio))
        if score < best_score:
            best_score = score
            best_idx = i
    return polys[best_idx]

def consensus_intersection_core(chain, tracker, buffer_dist=0.5, min_area=1e-3):
    polys = _chain_polygons_aligned(chain, tracker)
    if not polys:
        return None
    buffered = [p.buffer(buffer_dist) for p in polys]
    inter = reduce(lambda a, b: a.intersection(b), buffered)
    if inter.is_empty:
        return consensus_medoid(chain, tracker)
    core = _largest_polygon(inter.buffer(-buffer_dist))
    if core is None or core.is_empty or core.area < min_area:
        return consensus_medoid(chain, tracker)
    return core

def _buffer_to_target_area(poly, target_area, max_iters=30):
    if poly is None or poly.is_empty:
        return poly
    if poly.area <= target_area:
        return poly
    lo, hi = -max(1.0, poly.length / (2 * np.pi)), 0.0
    for _ in range(max_iters):
        mid = (lo + hi) / 2.0
        buff = poly.buffer(mid)
        area = buff.area if not buff.is_empty else 0.0
        if area > target_area:
            hi = mid
        else:
            lo = mid
    result = poly.buffer(hi)
    return result if not result.is_empty else poly

def consensus_union_shrink(chain, tracker):
    polys = _chain_polygons_aligned(chain, tracker)
    if not polys:
        return None
    union = _largest_polygon(unary_union(polys))
    target_area = float(np.median([p.area for p in polys]))
    return _buffer_to_target_area(union, target_area)

def extract_patch_for_polygon(om_id, polygon_aligned, tracker):
    if polygon_aligned is None or polygon_aligned.is_empty:
        return None
    dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
    polygon_original = translate(polygon_aligned, xoff=-dx, yoff=-dy)
    ortho_file = None
    for oid, (crown_file, of) in zip(tracker.om_ids, tracker.file_pairs):
        if oid == om_id:
            ortho_file = of
            break
    if not ortho_file or not os.path.exists(ortho_file):
        return None
    try:
        with rasterio.open(ortho_file) as src:
            out_image, _ = mask(src, [mapping(polygon_original)], crop=True)
            return np.moveaxis(out_image, 0, -1)
    except Exception:
        return None

def visualize_consensus_chain(chain, consensus_poly, tracker, title="Consensus Chain", show=True):
    if consensus_poly is None:
        print("No consensus polygon available.")
        return
    chain_length = len(chain)
    fig = plt.figure(figsize=(5 * chain_length, 10))
    print(f"\n{'='*80}")
    print(title)
    print(f"{'='*80}")
    print(f"Chain: {chain}")
    print(f"Length: {chain_length}\n")
    for idx, (om_id, crown_idx) in enumerate(chain):
        gdf = tracker.crowns_gdfs[om_id]
        crown = gdf.iloc[crown_idx]
        ax_poly = plt.subplot(2, chain_length, idx + 1)
        gdf.plot(ax=ax_poly, color='lightgray', edgecolor='gray', alpha=0.3)
        gpd.GeoSeries([consensus_poly]).plot(
            ax=ax_poly,
            facecolor='none',
            edgecolor='red',
            linewidth=2.5,
            alpha=0.9
        )
        gpd.GeoSeries([crown.geometry]).plot(
            ax=ax_poly,
            facecolor='none',
            edgecolor='black',
            linewidth=1.5,
            alpha=0.9
        )
        centroid = consensus_poly.centroid
        ax_poly.plot(centroid.x, centroid.y, 'o', color='yellow', markersize=8,
                     markeredgecolor='black', markeredgewidth=1.5)
        ax_poly.set_aspect('equal')
        ax_poly.set_title(f"OM{om_id} - Consensus Overlay", fontsize=10, fontweight='bold')
        ax_poly.grid(True, alpha=0.3)
        ax_img = plt.subplot(2, chain_length, chain_length + idx + 1)
        patch = extract_patch_for_polygon(om_id, consensus_poly, tracker)
        if patch is not None:
            ax_img.imshow(patch)
            ax_img.set_title(f"Consensus Patch\n{patch.shape[0]}×{patch.shape[1]} px", fontsize=9)
        else:
            ax_img.text(0.5, 0.5, 'Image not available', ha='center', va='center', fontsize=10, color='red')
            ax_img.set_title("No Image", fontsize=9)
        ax_img.axis('off')
    plt.tight_layout()
    if show:
        plt.show()
    print()

In [16]:
# # Compare normal chain visualization vs consensus crown strategies
# full_chains = chain_categories['full_width_1'] + chain_categories['full_branching']
# candidate_chains = full_chains if full_chains else chain_categories.get('partial_long', [])
# if not candidate_chains:
#     candidate_chains = chain_categories.get('partial_short', [])
# if not candidate_chains:
#     print("No chains available for consensus comparison.")
# else:
#     chain = candidate_chains[5]
#     print("\n=== Normal Chain Visualization ===")
#     visualize_chain_with_extracted_images(
#         chain,
#         tracker.crowns_gdfs,
#         tracker.crown_images,
#         tracker,
#         title="Original Crowns (per-OM)"
#     )
#     strategies = {
#         "medoid": consensus_medoid,
#         "intersection_core": consensus_intersection_core,
#         "union_shrink": consensus_union_shrink,
#     }
#     for name, func in strategies.items():
#         consensus_poly = func(chain, tracker)
#         visualize_consensus_chain(
#             chain,
#             consensus_poly,
#             tracker,
#             title=f"Consensus Strategy: {name}"
#         )

In [17]:
# # Visualize medoid consensus for all full chains without branching
# full_width_1_chains = chain_categories.get('full_width_1', [])
# print(f"Visualizing medoid consensus for {len(full_width_1_chains)} non-branching full chains...")
# if not full_width_1_chains:
#     print("No non-branching full chains found.")
# else:
#     for i, chain in enumerate(full_width_1_chains):
#         consensus_poly = consensus_medoid(chain, tracker)
#         visualize_consensus_chain(
#             chain,
#             consensus_poly,
#             tracker,
#             title=f"Medoid Consensus - Full Chain {i+1}/{len(full_width_1_chains)}"
#         )
#     print(f"✓ Completed medoid consensus visualization for {len(full_width_1_chains)} chains")

## Ground Truth Comparison

In [18]:
import json
from shapely.geometry import Polygon
from typing import Dict, List, Tuple

def _infer_pixel_space(seg_coords, width=None, height=None, pad=5.0):
    if width is None or height is None:
        return None
    if not seg_coords:
        return None
    xs = [p[0] for p in seg_coords]
    ys = [p[1] for p in seg_coords]
    max_x = max(xs)
    max_y = max(ys)
    # Likely pixel if coords fall within image bounds (with small padding)
    if max_x <= (width + pad) and max_y <= (height + pad):
        return True
    # Likely geo if far outside pixel bounds
    if max_x > (width * 2) or max_y > (height * 2):
        return False
    return None

def load_ground_truth_coco(json_path: str, ortho_path: str = None, transform_to_geo: str = "auto") -> gpd.GeoDataFrame:
    """
    Load ground truth crowns from COCO format JSON.
    
    Args:
        json_path: Path to COCO format JSON file
        ortho_path: Path to orthomosaic for coordinate transformation
        transform_to_geo: "auto", True, or False
    
    Returns:
        GeoDataFrame with ground truth crowns in geographic coordinates (if transformed)
    """
    with open(json_path, 'r') as f:
        coco_data = json.load(f)
    
    geometries = []
    gt_ids = []
    track_ids = []
    
    # Get transform from orthomosaic if needed
    transform = None
    ortho_crs = None
    ortho_w = None
    ortho_h = None
    if ortho_path and os.path.exists(ortho_path):
        with rasterio.open(ortho_path) as src:
            transform = src.transform
            ortho_crs = src.crs
            ortho_w = src.width
            ortho_h = src.height
            print(f"Orthomosaic transform: {transform}")
            print(f"Orthomosaic CRS: {src.crs}")
            print(f"Orthomosaic size: {src.width}×{src.height} px")
    
    # Try to infer pixel vs geo if auto
    if transform_to_geo == "auto":
        # Find first segmentation coords to infer scale
        sample_coords = None
        for ann in coco_data.get('annotations', []):
            if ann.get('segmentation') and len(ann['segmentation']) > 0:
                seg = ann['segmentation'][0]
                sample_coords = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
                break
        is_pixel = _infer_pixel_space(sample_coords or [], ortho_w, ortho_h)
        if is_pixel is None:
            transform_to_geo = bool(transform is not None)
        else:
            transform_to_geo = bool(is_pixel)
        print(f"Auto-detected ground truth coordinates as {'pixel' if transform_to_geo else 'geo'} space")
    
    for ann in coco_data['annotations']:
        # COCO segmentation is [[x1,y1,x2,y2,...]] in pixel coordinates
        seg = ann['segmentation'][0]
        pixel_coords = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
        
        if transform_to_geo and transform is not None:
            # Transform from pixel (col, row) to geographic (x, y)
            geo_coords = []
            for px, py in pixel_coords:
                # In rasterio, pixel (col, row) -> (x, y)
                x, y = rasterio.transform.xy(transform, py, px, offset='ul')
                geo_coords.append((x, y))
            poly = Polygon(geo_coords)
        else:
            # Keep as given coordinates
            poly = Polygon(pixel_coords)
        
        geometries.append(poly)
        gt_ids.append(ann['id'])
        track_ids.append(ann.get('attributes', {}).get('track_id', ann['id']))
    
    gdf = gpd.GeoDataFrame({
        'gt_id': gt_ids,
        'track_id': track_ids,
        'geometry': geometries
    }, crs=ortho_crs if ortho_crs is not None else None)
    
    print(f"Loaded {len(gdf)} ground truth crowns")
    if transform is not None and transform_to_geo:
        print(f"Transformed from pixel to geographic coordinates")
        # Print sample bounds to verify
        if len(gdf) > 0:
            bounds = gdf.total_bounds
            print(f"Bounds: x=[{bounds[0]:.1f}, {bounds[2]:.1f}], y=[{bounds[1]:.1f}, {bounds[3]:.1f}]")
    
    return gdf

def compute_metrics_at_iou_threshold(
    ground_truth: gpd.GeoDataFrame,
    predictions: gpd.GeoDataFrame,
    iou_threshold: float = 0.5
) -> Dict[str, float]:
    """
    Compute precision, recall, F1 score at a given IoU threshold.
    
    Args:
        ground_truth: GeoDataFrame with ground truth polygons
        predictions: GeoDataFrame with predicted/consensus polygons
        iou_threshold: Minimum IoU to count as a match
    
    Returns:
        Dictionary with precision, recall, f1, tp, fp, fn
    """
    gt_matched = set()
    pred_matched = set()
    
    # For each prediction, find best matching ground truth
    for pred_idx, pred_row in predictions.iterrows():
        pred_poly = pred_row.geometry
        if pred_poly is None or pred_poly.is_empty:
            continue
        
        best_iou = 0.0
        best_gt_idx = None
        
        for gt_idx, gt_row in ground_truth.iterrows():
            gt_poly = gt_row.geometry
            if gt_poly is None or gt_poly.is_empty:
                continue
            
            # Compute IoU
            try:
                intersection = pred_poly.intersection(gt_poly).area
                union = pred_poly.union(gt_poly).area
                iou = intersection / union if union > 0 else 0.0
                
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            except Exception:
                continue
        
        # If best IoU exceeds threshold, mark as match
        if best_iou >= iou_threshold and best_gt_idx is not None:
            gt_matched.add(best_gt_idx)
            pred_matched.add(pred_idx)
    
    tp = len(gt_matched)  # True positives: GT crowns matched
    fp = len(predictions) - len(pred_matched)  # False positives: predictions without GT match
    fn = len(ground_truth) - len(gt_matched)  # False negatives: GT crowns not detected
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'iou_threshold': iou_threshold
    }

def get_consensus_crowns_for_full_chains(
    tracker: 'TreeTrackingGraph',
    chain_categories: Dict,
    consensus_strategy = consensus_intersection_core
) -> gpd.GeoDataFrame:
    """
    Extract consensus crowns for all fully-tracked chains.
    
    Args:
        tracker: TreeTrackingGraph instance
        chain_categories: Dictionary with categorized chains
        consensus_strategy: Function to compute consensus polygon
    
    Returns:
        GeoDataFrame with consensus polygons for each full chain
    """
    full_chains = chain_categories.get('full_width_1', []) + chain_categories.get('full_branching', [])
    
    consensus_geoms = []
    chain_ids = []
    chain_lengths = []
    
    for i, chain in enumerate(full_chains):
        consensus_poly = consensus_strategy(chain, tracker)
        if consensus_poly is not None and not consensus_poly.is_empty:
            consensus_geoms.append(consensus_poly)
            chain_ids.append(i)
            chain_lengths.append(len(chain))
    
    base_crs = None
    if tracker and tracker.om_ids:
        base_crs = tracker.crowns_gdfs[tracker.om_ids[0]].crs
    gdf = gpd.GeoDataFrame({
        'chain_id': chain_ids,
        'chain_length': chain_lengths,
        'geometry': consensus_geoms
    }, crs=base_crs)
    
    print(f"Generated {len(gdf)} consensus crowns from {len(full_chains)} full chains")
    return gdf

print("Ground truth comparison functions loaded")

Ground truth comparison functions loaded


In [19]:
# Load ground truth with auto coordinate inference
gt_path = "../../input/ground_truth.json"
ortho_path = next((of for _, of in tracker.file_pairs if of), None)
ground_truth_gdf = load_ground_truth_coco(gt_path, ortho_path=ortho_path, transform_to_geo="auto")
print(f"Ground truth CRS: {ground_truth_gdf.crs}")

Orthomosaic transform: | 0.02, 0.00, 714263.73|
| 0.00,-0.02, 3159729.01|
| 0.00, 0.00, 1.00|
Orthomosaic CRS: PROJCS["WGS 84 / UTM zone 43N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",75],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Orthomosaic size: 10742×11102 px
Auto-detected ground truth coordinates as pixel space
Loaded 124 ground truth crowns
Transformed from pixel to geographic coordinates
Bounds: x=[714295.5, 714477.5], y=[3159492.0, 3159672.2]
Ground truth CRS: PROJCS["WGS 84 / UTM zone 43N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degre

In [20]:
# Generate consensus crowns for fully tracked chains using medoid strategy
consensus_crowns_gdf = get_consensus_crowns_for_full_chains(
    tracker, 
    chain_categories,
    consensus_strategy=consensus_medoid,
 )

# Ensure CRS is aligned (if ground truth has CRS)
if ground_truth_gdf.crs is not None and consensus_crowns_gdf.crs is None:
    consensus_crowns_gdf = consensus_crowns_gdf.set_crs(ground_truth_gdf.crs, allow_override=True)
elif ground_truth_gdf.crs is not None and consensus_crowns_gdf.crs != ground_truth_gdf.crs:
    consensus_crowns_gdf = consensus_crowns_gdf.to_crs(ground_truth_gdf.crs)

print(f"\nConsensus crowns: {len(consensus_crowns_gdf)}")
print(f"Ground truth crowns: {len(ground_truth_gdf)}")

Generated 20 consensus crowns from 20 full chains

Consensus crowns: 20
Ground truth crowns: 124


In [21]:
# Compute metrics at default IoU threshold (0.5)
metrics_050 = compute_metrics_at_iou_threshold(
    ground_truth_gdf,
    consensus_crowns_gdf,
    iou_threshold=0.3
)

print("\n" + "="*80)
print("GROUND TRUTH COMPARISON METRICS (IoU threshold = 0.3)")
print("="*80)
print(f"Precision:  {metrics_050['precision']:.3f}")
print(f"Recall:     {metrics_050['recall']:.3f}")
print(f"F1 Score:   {metrics_050['f1']:.3f}")
print(f"\nTrue Positives:  {metrics_050['tp']}")
print(f"False Positives: {metrics_050['fp']}")
print(f"False Negatives: {metrics_050['fn']}")
print("="*80)


GROUND TRUTH COMPARISON METRICS (IoU threshold = 0.3)
Precision:  1.000
Recall:     0.145
F1 Score:   0.254

True Positives:  18
False Positives: 0
False Negatives: 106


## Error Classification and Analysis

Classify different types of errors in ground truth vs consensus crown comparison.

In [22]:
from enum import Enum
from typing import Dict, List, Tuple, Optional
import pandas as pd

class ErrorType(Enum):
    """Classification of errors in ground truth vs prediction comparison."""
    # Detection errors
    MISSING_DETECTION = "Missing Detection"  # GT crown with no consensus match (FN)
    FALSE_POSITIVE = "False Positive"  # Consensus crown with no GT match (FP)
    
    # Geometric errors
    SPLIT_ERROR = "Split Error"  # 1 GT crown split into 2+ consensus crowns (1 GT -> multiple predictions)
    MERGE_ERROR = "Merge Error"  # 2+ GT crowns merged into 1 consensus crown (multiple GT -> 1 prediction)
    
    # Quality errors
    POOR_OVERLAP = "Poor Overlap"  # Match exists but IoU is low (below threshold)
    BOUNDARY_ERROR = "Boundary Error"  # IoU < threshold but has overlap, with size difference
    
    # Correct matches
    TRUE_POSITIVE = "True Positive"  # Good match (IoU >= threshold)

def classify_crown_errors(
    ground_truth: gpd.GeoDataFrame,
    predictions: gpd.GeoDataFrame,
    iou_threshold: float = 0.5,
    poor_overlap_threshold: float = 0.15,
    size_diff_threshold: float = 0.3,
    nearby_threshold: float = 15.0
) -> Tuple[List[Dict], pd.DataFrame]:
    """
    Classify errors between ground truth and predicted crowns.
    
    Split Error: 1 GT crown -> multiple predictions (we over-detected)
    Merge Error: multiple GT crowns -> 1 prediction (we under-detected)
    
    Args:
        ground_truth: GeoDataFrame with ground truth polygons
        predictions: GeoDataFrame with predicted/consensus polygons
        iou_threshold: Minimum IoU to count as true positive
        poor_overlap_threshold: Minimum IoU to consider as having overlap
        size_diff_threshold: Relative area difference for boundary errors
        nearby_threshold: Max distance for split/merge detection
    
    Returns:
        Tuple of (list of error cases, summary dataframe)
    """
    error_cases = []
    
    # Step 1: Build IoU matrix for ALL pairs
    iou_matrix = {}  # (gt_idx, pred_idx) -> iou
    
    for gt_idx, gt_row in ground_truth.iterrows():
        gt_poly = gt_row.geometry
        if gt_poly is None or gt_poly.is_empty:
            continue
        
        for pred_idx, pred_row in predictions.iterrows():
            pred_poly = pred_row.geometry
            if pred_poly is None or pred_poly.is_empty:
                continue
            
            try:
                intersection = gt_poly.intersection(pred_poly).area
                union = gt_poly.union(pred_poly).area
                iou = intersection / union if union > 0 else 0.0
                if iou > 0:
                    iou_matrix[(gt_idx, pred_idx)] = iou
            except Exception:
                continue
    
    # Step 2: Find matches for each GT and Pred
    gt_matches = {}  # gt_idx -> [(pred_idx, iou), ...]
    pred_matches = {}  # pred_idx -> [(gt_idx, iou), ...]
    
    for (gt_idx, pred_idx), iou in iou_matrix.items():
        if iou >= poor_overlap_threshold:
            if gt_idx not in gt_matches:
                gt_matches[gt_idx] = []
            gt_matches[gt_idx].append((pred_idx, iou))
            
            if pred_idx not in pred_matches:
                pred_matches[pred_idx] = []
            pred_matches[pred_idx].append((gt_idx, iou))
    
    # Sort matches by IoU (best first)
    for gt_idx in gt_matches:
        gt_matches[gt_idx].sort(key=lambda x: x[1], reverse=True)
    for pred_idx in pred_matches:
        pred_matches[pred_idx].sort(key=lambda x: x[1], reverse=True)
    
    # Step 3: Classify each GT crown
    gt_classified = set()
    
    for gt_idx, gt_row in ground_truth.iterrows():
        gt_poly = gt_row.geometry
        if gt_poly is None or gt_poly.is_empty:
            continue
        
        if gt_idx not in gt_matches:
            # No prediction overlaps with this GT
            error_cases.append({
                'error_type': ErrorType.MISSING_DETECTION.value,
                'gt_idx': int(gt_idx),
                'pred_idx': None,
                'iou': 0.0,
                'gt_area': float(gt_poly.area),
                'pred_area': None,
                'gt_centroid': gt_poly.centroid,
                'pred_centroid': None
            })
            gt_classified.add(gt_idx)
            continue
        
        matches = gt_matches[gt_idx]
        best_pred_idx, best_iou = matches[0]
        
        # Check for SPLIT: 1 GT -> multiple predictions
        if len(matches) > 1:
            # Check if multiple predictions are nearby and all have decent overlap
            pred_polys = [predictions.iloc[pred_idx].geometry for pred_idx, _ in matches]
            nearby_preds = []
            for i, (pred_idx, iou) in enumerate(matches):
                pred_poly = predictions.iloc[pred_idx].geometry
                # Check if this prediction is near the GT centroid
                dist = gt_poly.centroid.distance(pred_poly.centroid)
                if dist < nearby_threshold and iou >= poor_overlap_threshold:
                    nearby_preds.append((pred_idx, iou))
            
            if len(nearby_preds) > 1:
                # SPLIT ERROR: This GT crown was split into multiple predictions
                error_cases.append({
                    'error_type': ErrorType.SPLIT_ERROR.value,
                    'gt_idx': int(gt_idx),
                    'pred_idx': [int(idx) for idx, _ in nearby_preds],
                    'iou': [float(iou) for _, iou in nearby_preds],
                    'gt_area': float(gt_poly.area),
                    'pred_area': [float(predictions.iloc[idx].geometry.area) for idx, _ in nearby_preds],
                    'gt_centroid': gt_poly.centroid,
                    'pred_centroid': [predictions.iloc[idx].geometry.centroid for idx, _ in nearby_preds],
                    'num_splits': len(nearby_preds)
                })
                gt_classified.add(gt_idx)
                continue
        
        # Single best match - classify based on IoU
        pred_poly = predictions.iloc[best_pred_idx].geometry
        
        if best_iou >= iou_threshold:
            # TRUE POSITIVE: Good match
            error_cases.append({
                'error_type': ErrorType.TRUE_POSITIVE.value,
                'gt_idx': int(gt_idx),
                'pred_idx': int(best_pred_idx),
                'iou': float(best_iou),
                'gt_area': float(gt_poly.area),
                'pred_area': float(pred_poly.area),
                'gt_centroid': gt_poly.centroid,
                'pred_centroid': pred_poly.centroid
            })
        elif best_iou >= poor_overlap_threshold:
            # Check for boundary error (has overlap but low IoU, with size difference)
            area_ratio = abs(gt_poly.area - pred_poly.area) / max(gt_poly.area, pred_poly.area)
            if area_ratio > size_diff_threshold:
                error_type = ErrorType.BOUNDARY_ERROR.value
            else:
                error_type = ErrorType.POOR_OVERLAP.value
            
            error_cases.append({
                'error_type': error_type,
                'gt_idx': int(gt_idx),
                'pred_idx': int(best_pred_idx),
                'iou': float(best_iou),
                'gt_area': float(gt_poly.area),
                'pred_area': float(pred_poly.area),
                'gt_centroid': gt_poly.centroid,
                'pred_centroid': pred_poly.centroid,
                'area_ratio_diff': float(area_ratio) if area_ratio else None
            })
        else:
            # Very poor overlap, treat as missing
            error_cases.append({
                'error_type': ErrorType.MISSING_DETECTION.value,
                'gt_idx': int(gt_idx),
                'pred_idx': None,
                'iou': 0.0,
                'gt_area': float(gt_poly.area),
                'pred_area': None,
                'gt_centroid': gt_poly.centroid,
                'pred_centroid': None
            })
        
        gt_classified.add(gt_idx)
    
    # Step 4: Classify predictions for MERGE errors and FALSE POSITIVES
    pred_classified = set()
    
    for pred_idx, pred_row in predictions.iterrows():
        pred_poly = pred_row.geometry
        if pred_poly is None or pred_poly.is_empty:
            continue
        
        if pred_idx not in pred_matches:
            # FALSE POSITIVE: No GT overlaps with this prediction
            error_cases.append({
                'error_type': ErrorType.FALSE_POSITIVE.value,
                'gt_idx': None,
                'pred_idx': int(pred_idx),
                'iou': 0.0,
                'gt_area': None,
                'pred_area': float(pred_poly.area),
                'gt_centroid': None,
                'pred_centroid': pred_poly.centroid
            })
            pred_classified.add(pred_idx)
            continue
        
        matches = pred_matches[pred_idx]
        
        # Check for MERGE: multiple GT -> 1 prediction
        if len(matches) > 1:
            # Check if multiple GTs are nearby and all have decent overlap
            nearby_gts = []
            for gt_idx, iou in matches:
                gt_poly = ground_truth.iloc[gt_idx].geometry
                dist = pred_poly.centroid.distance(gt_poly.centroid)
                if dist < nearby_threshold and iou >= poor_overlap_threshold:
                    nearby_gts.append((gt_idx, iou))
            
            if len(nearby_gts) > 1:
                # MERGE ERROR: Multiple GT crowns merged into this prediction
                error_cases.append({
                    'error_type': ErrorType.MERGE_ERROR.value,
                    'gt_idx': [int(idx) for idx, _ in nearby_gts],
                    'pred_idx': int(pred_idx),
                    'iou': [float(iou) for _, iou in nearby_gts],
                    'gt_area': [float(ground_truth.iloc[idx].geometry.area) for idx, _ in nearby_gts],
                    'pred_area': float(pred_poly.area),
                    'gt_centroid': [ground_truth.iloc[idx].geometry.centroid for idx, _ in nearby_gts],
                    'pred_centroid': pred_poly.centroid,
                    'num_merged': len(nearby_gts)
                })
                pred_classified.add(pred_idx)
        # If not a merge and not already classified as FP, it was classified in GT loop
    
    # Create summary dataframe
    error_types = [case['error_type'] for case in error_cases]
    summary = pd.DataFrame({
        'Error Type': list(ErrorType.__members__.values()),
        'Count': [error_types.count(et.value) for et in ErrorType]
    })
    
    return error_cases, summary
    
    return error_cases, summary

print("✓ Error classification functions loaded")

✓ Error classification functions loaded


In [23]:
# Classify errors for IoU threshold 0.3
error_cases_03, error_summary_03 = classify_crown_errors(
    ground_truth_gdf,
    consensus_crowns_gdf,
    iou_threshold=0.3,
    poor_overlap_threshold=0.15,
    size_diff_threshold=0.3,
    nearby_threshold=15.0
)

print("\n" + "="*80)
print("ERROR CLASSIFICATION SUMMARY (IoU threshold = 0.3)")
print("="*80)
print(error_summary_03.to_string(index=False))
print("="*80)
print(f"\nTotal cases analyzed: {len(error_cases_03)}")


ERROR CLASSIFICATION SUMMARY (IoU threshold = 0.3)
                 Error Type  Count
ErrorType.MISSING_DETECTION    103
   ErrorType.FALSE_POSITIVE      0
      ErrorType.SPLIT_ERROR      2
      ErrorType.MERGE_ERROR      3
     ErrorType.POOR_OVERLAP      0
   ErrorType.BOUNDARY_ERROR      2
    ErrorType.TRUE_POSITIVE     17

Total cases analyzed: 127


In [24]:
# Also classify for IoU threshold 0.5
error_cases_05, error_summary_05 = classify_crown_errors(
    ground_truth_gdf,
    consensus_crowns_gdf,
    iou_threshold=0.5,
    poor_overlap_threshold=0.25,
    size_diff_threshold=0.3,
    nearby_threshold=15.0
)

print("\n" + "="*80)
print("ERROR CLASSIFICATION SUMMARY (IoU threshold = 0.5)")
print("="*80)
print(error_summary_05.to_string(index=False))
print("="*80)
print(f"\nTotal cases analyzed: {len(error_cases_05)}")


ERROR CLASSIFICATION SUMMARY (IoU threshold = 0.5)
                 Error Type  Count
ErrorType.MISSING_DETECTION    105
   ErrorType.FALSE_POSITIVE      0
      ErrorType.SPLIT_ERROR      2
      ErrorType.MERGE_ERROR      1
     ErrorType.POOR_OVERLAP      2
   ErrorType.BOUNDARY_ERROR      6
    ErrorType.TRUE_POSITIVE      9

Total cases analyzed: 125


In [25]:
def visualize_error_case(
    error_case: Dict,
    ground_truth: gpd.GeoDataFrame,
    predictions: gpd.GeoDataFrame,
    ortho_path: str,
    title: str = "Error Case",
    figsize: Tuple[int, int] = (15, 5)
):
    """
    Visualize a specific error case with polygon overlay and RGB image.
    
    Args:
        error_case: Dictionary with error case information
        ground_truth: GeoDataFrame with ground truth polygons
        predictions: GeoDataFrame with predicted polygons
        ortho_path: Path to orthomosaic for background image
        title: Plot title
        figsize: Figure size
    """
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    error_type = error_case['error_type']
    gt_idx = error_case.get('gt_idx')
    pred_idx = error_case.get('pred_idx')
    
    # Determine extent for visualization
    gt_polys = []
    pred_polys = []
    
    if gt_idx is not None:
        if isinstance(gt_idx, list):
            gt_polys = [ground_truth.iloc[idx].geometry for idx in gt_idx]
        else:
            gt_polys = [ground_truth.iloc[gt_idx].geometry]
    
    if pred_idx is not None:
        if isinstance(pred_idx, list):
            pred_polys = [predictions.iloc[idx].geometry for idx in pred_idx]
        else:
            pred_polys = [predictions.iloc[pred_idx].geometry]
    
    all_polys = []
    for p in gt_polys + pred_polys:
        if p is not None and hasattr(p, 'is_empty') and not p.is_empty:
            all_polys.append(p)
    
    if not all_polys:
        print(f"No valid polygons for case: {error_case}")
        plt.close(fig)
        return
    
    # Get bounds with margin
    union_poly = unary_union(all_polys)
    minx, miny, maxx, maxy = union_poly.bounds
    width = maxx - minx
    height = maxy - miny
    margin = max(width, height) * 0.3
    
    extent = [minx - margin, maxx + margin, miny - margin, maxy + margin]
    
    # Load orthomosaic for background
    ortho_img = None
    try:
        with rasterio.open(ortho_path) as src:
            # Transform extent to pixel coordinates
            window = rasterio.windows.from_bounds(
                extent[0], extent[2], extent[1], extent[3], 
                src.transform
            )
            ortho_img = src.read([1, 2, 3], window=window)
            ortho_img = np.moveaxis(ortho_img, 0, -1)
            # Normalize for display
            ortho_img = np.clip(ortho_img / ortho_img.max() * 255, 0, 255).astype(np.uint8)
    except Exception as e:
        print(f"Could not load ortho image: {e}")
    
    # Plot 1: Ground truth only
    ax1 = axes[0]
    if ortho_img is not None:
        ax1.imshow(ortho_img, extent=extent, aspect='auto')
    for gt_poly in gt_polys:
        gpd.GeoSeries([gt_poly]).plot(
            ax=ax1, facecolor='none', edgecolor='blue', linewidth=2.5, alpha=0.9
        )
    ax1.set_xlim(extent[0], extent[1])
    ax1.set_ylim(extent[2], extent[3])
    ax1.set_title("Ground Truth", fontsize=10, fontweight='bold')
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Prediction only
    ax2 = axes[1]
    if ortho_img is not None:
        ax2.imshow(ortho_img, extent=extent, aspect='auto')
    for pred_poly in pred_polys:
        gpd.GeoSeries([pred_poly]).plot(
            ax=ax2, facecolor='none', edgecolor='red', linewidth=2.5, alpha=0.9
        )
    ax2.set_xlim(extent[0], extent[1])
    ax2.set_ylim(extent[2], extent[3])
    ax2.set_title("Consensus Crown", fontsize=10, fontweight='bold')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Overlay
    ax3 = axes[2]
    if ortho_img is not None:
        ax3.imshow(ortho_img, extent=extent, aspect='auto')
    for gt_poly in gt_polys:
        gpd.GeoSeries([gt_poly]).plot(
            ax=ax3, facecolor='blue', edgecolor='blue', linewidth=2, alpha=0.3, label='GT'
        )
    for pred_poly in pred_polys:
        gpd.GeoSeries([pred_poly]).plot(
            ax=ax3, facecolor='red', edgecolor='red', linewidth=2, alpha=0.3, label='Consensus'
        )
    ax3.set_xlim(extent[0], extent[1])
    ax3.set_ylim(extent[2], extent[3])
    ax3.set_title("Overlay", fontsize=10, fontweight='bold')
    ax3.set_aspect('equal')
    ax3.grid(True, alpha=0.3)
    
    # Add handles for legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='blue', edgecolor='blue', alpha=0.5, label='Ground Truth'),
        Patch(facecolor='red', edgecolor='red', alpha=0.5, label='Consensus')
    ]
    ax3.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    # Main title with case details
    iou_str = f"{error_case.get('iou', 0):.3f}" if isinstance(error_case.get('iou'), (int, float)) else "N/A"
    fig.suptitle(f"{title}: {error_type} (IoU: {iou_str})", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def visualize_error_examples(
    error_cases: List[Dict],
    ground_truth: gpd.GeoDataFrame,
    predictions: gpd.GeoDataFrame,
    ortho_path: str,
    max_per_type: int = 2
):
    """
    Visualize examples of each error type.
    
    Args:
        error_cases: List of error case dictionaries
        ground_truth: GeoDataFrame with ground truth polygons
        predictions: GeoDataFrame with predicted polygons
        ortho_path: Path to orthomosaic
        max_per_type: Maximum number of examples per error type
    """
    # Group cases by error type
    by_type = {}
    for case in error_cases:
        error_type = case['error_type']
        if error_type not in by_type:
            by_type[error_type] = []
        by_type[error_type].append(case)
    
    # Visualize examples of each type
    for error_type, cases in by_type.items():
        if len(cases) == 0:
            continue
        
        print(f"\n{'='*80}")
        print(f"{error_type.upper()} - {len(cases)} cases")
        print(f"{'='*80}\n")
        
        # Show up to max_per_type examples
        for i, case in enumerate(cases[:max_per_type]):
            visualize_error_case(
                case,
                ground_truth,
                predictions,
                ortho_path,
                title=f"{error_type} Example {i+1}/{min(len(cases), max_per_type)}",
                figsize=(15, 5)
            )

print("✓ Error visualization functions loaded")

✓ Error visualization functions loaded


In [40]:
# # Visualize examples of each error type (IoU 0.3)
# visualize_error_examples(
#     error_cases_03,
#     ground_truth_gdf,
#     consensus_crowns_gdf,
#     ortho_path,
#     max_per_type=3
# )

## Conditional Metrics (Excluding Specific Error Types)

Calculate precision and recall after excluding certain error categories to better understand model performance.

In [27]:
def compute_conditional_metrics(
    error_cases: List[Dict],
    exclude_types: List[str] = None
) -> Dict[str, float]:
    """
    Compute precision, recall, F1 excluding specific error types.
    
    This allows us to understand performance under different assumptions:
    - Excluding MISSING_DETECTION: How good are we when we do detect something?
    - Excluding FALSE_POSITIVE: How good are matches when we ignore extra detections?
    - Excluding SPLIT/MERGE: Performance on simple 1-to-1 matches
    
    Args:
        error_cases: List of classified error cases
        exclude_types: List of error type names to exclude from calculation
    
    Returns:
        Dictionary with conditional metrics
    """
    if exclude_types is None:
        exclude_types = []
    
    # Filter cases
    filtered_cases = [
        case for case in error_cases 
        if case['error_type'] not in exclude_types
    ]
    
    # Count by type in filtered set
    tp = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.TRUE_POSITIVE.value)
    fp = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.FALSE_POSITIVE.value)
    fn = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.MISSING_DETECTION.value)
    
    # Other errors that affect counts
    split_errors = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.SPLIT_ERROR.value)
    merge_errors = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.MERGE_ERROR.value)
    poor_overlap = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.POOR_OVERLAP.value)
    boundary_errors = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.BOUNDARY_ERROR.value)
    
    # Depending on interpretation, boundary errors could be TP
    # and poor_overlap, split, merge could be FP
    total_detected = tp + fp + split_errors + merge_errors + poor_overlap + boundary_errors
    total_gt = tp + fn + split_errors + merge_errors + poor_overlap + boundary_errors
    
    precision = tp / total_detected if total_detected > 0 else 0.0
    recall = tp / total_gt if total_gt > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'split_errors': split_errors,
        'merge_errors': merge_errors,
        'poor_overlap': poor_overlap,
        'boundary_errors': boundary_errors,
        'excluded_types': exclude_types,
        'total_cases': len(filtered_cases)
    }

def create_conditional_metrics_table(error_cases: List[Dict], iou_threshold: float) -> pd.DataFrame:
    """
    Create a table of metrics under different exclusion scenarios.
    
    Args:
        error_cases: List of error cases
        iou_threshold: IoU threshold used for classification
    
    Returns:
        DataFrame with conditional metrics
    """
    scenarios = [
        ("All Errors Included", []),
        ("Exclude False Positives", [ErrorType.FALSE_POSITIVE.value]),
        ("Exclude Split Errors", [ErrorType.SPLIT_ERROR.value]),
        ("Exclude Merge Errors", [ErrorType.MERGE_ERROR.value]),
        ("Exclude Poor Overlap", [ErrorType.POOR_OVERLAP.value]),
        ("Exclude Boundary Errors", [ErrorType.BOUNDARY_ERROR.value]),
        ("Exclude Split & Merge", [ErrorType.SPLIT_ERROR.value, ErrorType.MERGE_ERROR.value]),
        ("Exclude Detection Errors", [ErrorType.MISSING_DETECTION.value, ErrorType.FALSE_POSITIVE.value]),
        ("Exclude All Geometric Errors", [ErrorType.SPLIT_ERROR.value, ErrorType.MERGE_ERROR.value, 
                                          ErrorType.POOR_OVERLAP.value, ErrorType.BOUNDARY_ERROR.value]),
    ]
    
    rows = []
    for scenario_name, exclude_types in scenarios:
        metrics = compute_conditional_metrics(error_cases, exclude_types)
        rows.append({
            'Scenario': scenario_name,
            'Precision': f"{metrics['precision']:.3f}",
            'Recall': f"{metrics['recall']:.3f}",
            'F1': f"{metrics['f1']:.3f}",
            'TP': metrics['tp'],
            'FP': metrics['fp'],
            'FN': metrics['fn'],
            'Cases': metrics['total_cases']
        })
    
    df = pd.DataFrame(rows)
    return df

print("✓ Conditional metrics functions loaded")

✓ Conditional metrics functions loaded


In [28]:
# Create conditional metrics table for IoU 0.3
conditional_metrics_03 = create_conditional_metrics_table(error_cases_03, iou_threshold=0.3)

print("\n" + "="*80)
print("CONDITIONAL METRICS TABLE (IoU threshold = 0.3)")
print("="*80)
print(conditional_metrics_03.to_string(index=False))
print("="*80)


CONDITIONAL METRICS TABLE (IoU threshold = 0.3)
                    Scenario Precision Recall    F1  TP  FP  FN  Cases
         All Errors Included     0.708  0.134 0.225  17   0 103    127
     Exclude False Positives     0.708  0.134 0.225  17   0 103    127
        Exclude Split Errors     0.773  0.136 0.231  17   0 103    125
        Exclude Merge Errors     0.810  0.137 0.234  17   0 103    124
        Exclude Poor Overlap     0.708  0.134 0.225  17   0 103    127
     Exclude Boundary Errors     0.773  0.136 0.231  17   0 103    125
       Exclude Split & Merge     0.895  0.139 0.241  17   0 103    122
    Exclude Detection Errors     0.708  0.708 0.708  17   0   0     24
Exclude All Geometric Errors     1.000  0.142 0.248  17   0 103    120


In [29]:
# Create conditional metrics table for IoU 0.5
conditional_metrics_05 = create_conditional_metrics_table(error_cases_05, iou_threshold=0.5)

print("\n" + "="*80)
print("CONDITIONAL METRICS TABLE (IoU threshold = 0.5)")
print("="*80)
print(conditional_metrics_05.to_string(index=False))
print("="*80)


CONDITIONAL METRICS TABLE (IoU threshold = 0.5)
                    Scenario Precision Recall    F1  TP  FP  FN  Cases
         All Errors Included     0.450  0.072 0.124   9   0 105    125
     Exclude False Positives     0.450  0.072 0.124   9   0 105    125
        Exclude Split Errors     0.500  0.073 0.128   9   0 105    123
        Exclude Merge Errors     0.474  0.073 0.126   9   0 105    124
        Exclude Poor Overlap     0.500  0.073 0.128   9   0 105    123
     Exclude Boundary Errors     0.643  0.076 0.135   9   0 105    119
       Exclude Split & Merge     0.529  0.074 0.129   9   0 105    122
    Exclude Detection Errors     0.450  0.450 0.450   9   0   0     20
Exclude All Geometric Errors     1.000  0.079 0.146   9   0 105    114


## Detailed Error Type Examples

Now let's visualize specific examples of each major error type to understand what's happening.

In [33]:
# # Visualize specific examples of key error types
# key_error_types = [
#     ErrorType.TRUE_POSITIVE.value,
#     ErrorType.SPLIT_ERROR.value,
#     ErrorType.MERGE_ERROR.value,
#     ErrorType.MISSING_DETECTION.value,
# ]

# for error_type in key_error_types:
#     type_cases = [case for case in error_cases_03 if case['error_type'] == error_type]
#     if len(type_cases) > 0:
#         print(f"\n{'='*80}")
#         print(f"EXAMPLES: {error_type.upper()} ({len(type_cases)} total cases)")
#         print(f"{'='*80}\n")
        
#         # Show 2 examples of each type
#         for i, case in enumerate(type_cases[:2]):
#             visualize_error_case(
#                 case,
#                 ground_truth_gdf,
#                 consensus_crowns_gdf,
#                 ortho_path,
#                 title=f"{error_type} - Example {i+1}",
#                 figsize=(15, 5)
#             )

In [ ]:
# # Visualize overlap: consensus crowns vs ground truth (OM1 coordinate system)
# import matplotlib.pyplot as plt

# def _sample_for_plot(gdf, max_items=200, seed=7):
#     if len(gdf) <= max_items:
#         return gdf
#     return gdf.sample(n=max_items, random_state=seed)

# def plot_gt_vs_consensus_overlap(gt_gdf, cons_gdf, title="GT vs Consensus Overlap", max_items=200):
#     if gt_gdf is None or cons_gdf is None or gt_gdf.empty or cons_gdf.empty:
#         print("No data to plot.")
#         return
#     gt = _sample_for_plot(gt_gdf, max_items=max_items)
#     cons = _sample_for_plot(cons_gdf, max_items=max_items)
#     fig, ax = plt.subplots(1, 1, figsize=(10, 10))
#     gt.plot(ax=ax, facecolor='none', edgecolor='blue', linewidth=1.0, alpha=0.6, label='Ground Truth')
#     cons.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=1.2, alpha=0.7, label='Consensus')
#     ax.set_aspect('equal')
#     ax.set_title(title)
#     ax.grid(True, alpha=0.2)
#     # Build legend manually for geopandas
#     from matplotlib.lines import Line2D
#     legend_elements = [
#         Line2D([0], [0], color='blue', lw=2, label='Ground Truth'),
#         Line2D([0], [0], color='red', lw=2, label='Consensus'),
#     ]
#     ax.legend(handles=legend_elements, loc='best')
#     plt.tight_layout()
#     plt.show()

# plot_gt_vs_consensus_overlap(ground_truth_gdf, consensus_crowns_gdf, title="GT vs Consensus Crowns (OM1/UTM)")

## 10. Sweep: Similarity Threshold vs Full-Length Chains
This sweeps a single scalar that globally scales the per-case `similarity_threshold` values used in `build_graph_conditional()` (keeping the conditional matching framework intact).
We then count how many full-length (5-OM) chains are recovered at each setting and plot the result.

In [35]:
# import contextlib
# import io
# from dataclasses import replace
# from pathlib import Path
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from IPython.display import display

# def _set_case_similarity_thresholds(case_configs, threshold, clip=(0.0, 1.0)):
#     """Return a new dict of case configs with similarity_threshold set to `threshold` for all cases."""
#     lo, hi = clip
#     thr = float(np.clip(threshold, lo, hi))
#     return {name: replace(cfg, similarity_threshold=thr) for name, cfg in case_configs.items()}

# def _count_full_length_chains(tracker):
#     cats = categorize_chains(tracker)
#     full_width_1 = len(cats.get('full_width_1', []))
#     full_branching = len(cats.get('full_branching', []))
#     return {
#         'full_width_1': full_width_1,
#         'full_branching': full_branching,
#         'full_total': full_width_1 + full_branching,
#     }

# def sweep_similarity_threshold_vs_full_chains(
#     tracker,
#     thresholds=None,
#     *,
#     base_max_dist=140.0,
#     overlap_gate=0.02,
#     min_base_similarity=0.0,
#     max_candidates_per_prev=8,
#     max_candidates_per_curr=8,
#     silent=True,
#     case_order=None,
#     classify_mode=None,
#     save_csv=True,
#     csv_name='similarity_threshold_sweep_full_chains.csv',
# ):
#     """
#     Sweep a single absolute similarity threshold (applied to all match cases) and measure full-chain counts.
    
#     Why this is a good research knob:
#     - It keeps your conditional case logic, scoring features, and mutual-best selection intact.
#     - It only changes how strict you are about accepting a scored candidate edge.
#     """
#     if thresholds is None:
#         thresholds = list(np.round(np.arange(0.0, 0.50 + 1e-9, 0.05), 2))
    
#     order = case_order or getattr(tracker, 'case_order', None) or list(tracker.case_configs.keys())
#     mode = classify_mode or getattr(tracker, 'match_case_mode', None)
#     base_case_configs = tracker.case_configs
    
#     results = []
#     for thr in thresholds:
#         thr = float(thr)
#         fixed_configs = _set_case_similarity_thresholds(base_case_configs, thr)
#         buf = io.StringIO()
#         with contextlib.redirect_stdout(buf) if silent else contextlib.nullcontext():
#             tracker.build_graph_conditional(
#                 base_max_dist=base_max_dist,
#                 overlap_gate=overlap_gate,
#                 min_base_similarity=min_base_similarity,
#                 max_candidates_per_prev=max_candidates_per_prev,
#                 max_candidates_per_curr=max_candidates_per_curr,
#                 classify_mode=mode,
#                 case_configs=fixed_configs,
#                 case_order=order,
#             )
#         counts = _count_full_length_chains(tracker)
#         results.append({
#             'similarity_threshold': thr,
#             'edges': int(tracker.G.number_of_edges()),
#             **counts,
#         })
#     df = pd.DataFrame(results)
#     if save_csv:
#         out_dir = Path(getattr(tracker, 'output_dir', '.'))
#         out_dir.mkdir(parents=True, exist_ok=True)
#         out_path = out_dir / csv_name
#         df.to_csv(out_path, index=False)
#         print(f"Saved sweep results to: {out_path}")
#     return df

# # --- Run the requested sweep: 0.00 → 0.50 (step 0.05) ---
# df_sweep = sweep_similarity_threshold_vs_full_chains(
#     tracker,
#     thresholds=np.round(np.arange(0.0, 0.80 + 1e-9, 0.05), 2).tolist(),
#     silent=True,
#     save_csv=True,
#     csv_name='similarity_threshold_0_to_0p5_step_0p05_full_chains.csv',
# )
# display(df_sweep)

# fig, ax = plt.subplots(figsize=(9, 5))
# x = df_sweep['similarity_threshold']
# ax.plot(x, df_sweep['full_total'], marker='o', linewidth=2, label='Full chains (total)')
# ax.plot(x, df_sweep['full_width_1'], marker='o', linewidth=2, label='Full chains (no branching)')
# ax.plot(x, df_sweep['full_branching'], marker='o', linewidth=2, label='Full chains (branching)')
# ax.set_xlabel('Similarity threshold (applied to all cases)')
# ax.set_ylabel('Count of full-length chains (length = 5)')
# ax.set_title('Sweep: Similarity Threshold vs Full-Length Chains')
# ax.grid(True, alpha=0.3)
# ax.legend(loc='best')

# ax2 = ax.twinx()
# ax2.plot(x, df_sweep['edges'], linestyle='--', color='gray', alpha=0.7)
# ax2.set_ylabel('Graph edges (diagnostic)')

# best_idx = int(df_sweep['full_total'].idxmax())
# best_row = df_sweep.loc[best_idx]
# ax.scatter([best_row['similarity_threshold']], [best_row['full_total']], s=90, zorder=5)
# ax.annotate(
#     f"best full_total={int(best_row['full_total'])}\nthr={best_row['similarity_threshold']:.2f}",
#     (best_row['similarity_threshold'], best_row['full_total']),
#     textcoords="offset points",
#     xytext=(10, 10),
#     ha='left',
#     fontsize=9,
#     bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='0.7', alpha=0.9),
# )

# plt.tight_layout()
# plt.show()